# LeetCode 411 — Minimum Unique Word Abbreviation bằng Python

> **Lưu ý: đây là bài Premium** (bài bị khóa trên LeetCode, chỉ tài khoản trả phí mới mở được). Phát biểu đề, ví dụ, constraints và signature dưới đây được viết lại theo bản LeetCode mà mình biết. Chỗ nào mình không chắc 100% sẽ được ghi rõ là **giả định**.
> Độ khó trên LeetCode: **Hard**.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minAbbreviation(self, target: str, dictionary: List[str]) -> str:

```

Bài này là "bài cuối" của một chuỗi 3 bài về **viết tắt từ** (word abbreviation):

| Bài | Hỏi gì | Vai trò trong notebook |
|---|---|---|
| #408 Valid Word Abbreviation (Easy) | Chuỗi `abbr` có phải viết tắt của `word` không? | Dùng làm **hàm kiểm tra** (trọng tài) |
| #320 Generalized Abbreviation (Medium) | Liệt kê **mọi** viết tắt của một từ | Dạy cách sinh viết tắt bằng **bitmask** |
| **#411** (Hard) | Viết tắt **ngắn nhất** của `target` mà **không** là viết tắt của từ nào trong `dictionary` | Bài chính |

## Sau notebook này, bạn sẽ nắm được

1. Đề bài yêu cầu gì: viết tắt là gì, vì sao hai khối số không được **kề nhau**, và "độ dài viết tắt" được đếm thế nào (`"s10n"` dài **3**, không phải 4).
2. Ôn từ căn bản **#408** (kiểm tra viết tắt bằng hai con trỏ) và **#320** (sinh mọi viết tắt). Học **bitmask** từ đầu: bit, `<<`, `>>`, `&`, `|`, `~`, `bit_count()`, duyệt mọi tập con bằng `range(1 << m)`.
3. Bốn cách giải, đi từ trực giác đến khéo léo:
   - **Cách 1:** Brute force — sinh cả `2^m` viết tắt dạng chuỗi, sắp theo độ dài, kiểm tra từng cái với từng từ bằng hàm #408. Chậm nhưng chắc chắn đúng → dùng làm **trọng tài** khi test.
   - **Cách 2 (cách chính):** **Diff-mask** — chỉ giữ từ **cùng độ dài**; mỗi từ cho một mask "các vị trí khác `target`"; một viết tắt (mask "giữ chữ") hợp lệ ⇔ **giao (`&`) với mọi diff-mask đều khác 0**. Kèm công thức tính **độ dài viết tắt từ mask** chỉ bằng vài phép bit.
   - **Cách 3:** **DFS + cắt tỉa** (branch and bound) — dựng viết tắt từ trái sang phải, bỏ ngay nhánh đã dài hơn đáp án tốt nhất hoặc đã "chết chắc".
   - **Cách 4 (tùy chọn):** Chỉ duyệt **mask con** của hợp các diff-mask và bỏ các diff-mask thừa.
4. Trace (mô phỏng) từng bước, code nộp LeetCode, cách test khi **có nhiều đáp án đúng** (kiểm tra tính hợp lệ + so độ dài với brute force), lỗi thường gặp, bài tập + đáp án.

## 1. Đề bài

Một chuỗi có thể được **viết tắt** (abbreviated) bằng cách thay **một số bất kỳ** các **chuỗi con không kề nhau** (non-adjacent substrings) bằng **độ dài** của chúng. Ví dụ, chuỗi `"substitution"` có thể được viết tắt thành (không chỉ có các cách này):

- `"s10n"` (`"s ubstitutio n"`)
- `"sub4u4"` (`"sub stit u tion"`)
- `"12"` (`"substitution"`)
- `"su3i1u2on"` (`"su bst i t u ti on"`)
- `"substitution"` (không thay chuỗi con nào)

Lưu ý `"s55n"` (`"s ubsti tutio n"`) **không** phải viết tắt hợp lệ của `"substitution"` vì hai chuỗi con bị thay nằm **kề nhau**.

**Độ dài** của một viết tắt = **số chữ cái không bị thay** + **số chuỗi con bị thay**. Ví dụ `"s10n"` có độ dài `3` (2 chữ cái + 1 chuỗi con) và `"su3i1u2on"` có độ dài `9` (6 chữ cái + 3 chuỗi con).

Cho chuỗi `target` và mảng chuỗi `dictionary`, trả về **một viết tắt của `target` có độ dài nhỏ nhất có thể** sao cho nó **không phải là viết tắt của bất kỳ chuỗi nào** trong `dictionary`. Nếu có nhiều viết tắt ngắn nhất, trả về **bất kỳ** cái nào.

**Ví dụ 1**

```text
Input:  target = "apple", dictionary = ["blade"]
Output: "a4"
Explanation: The shortest abbreviation of "apple" is "5", but this is also an abbreviation of "blade".
The next shortest abbreviations are "a4" and "4e". "4e" is an abbreviation of blade while "a4" is not.
Hence, return "a4".
```

**Ví dụ 2**

```text
Input:  target = "apple", dictionary = ["blade","plain","amber"]
Output: "1p3"
Explanation: "5" is an abbreviation of both "apple" but also every word in the dictionary.
"a4" is an abbreviation of "apple" but also "amber".
"4e" is an abbreviation of "apple" but also "blade".
"1p3", "2p2", and "3l1" are the next shortest abbreviations of "apple".
Since none of them are abbreviations of words in the dictionary, returning any of them is correct.
```

**Constraints (ràng buộc):**

- `m == target.length`
- `n == dictionary.length`
- `1 <= m <= 21`
- `0 <= n <= 1000`
- `1 <= dictionary[i].length <= 100`
- `log2(n) + m <= 21` nếu `n > 0`
- `target` và `dictionary[i]` chỉ gồm chữ cái tiếng Anh viết thường.
- `dictionary` **không** chứa `target`.

> Ghi chú về độ chắc chắn: đề và hai ví dụ ở trên mình nhớ khá chắc. Với constraints, các dòng `m <= 21`, `log2(n) + m <= 21` và "`dictionary` không chứa `target`" là phần quan trọng nhất; nếu bản LeetCode hiện tại có khác chút ít (ví dụ cận trên của `n`), lời giải trong notebook **vẫn đúng**, vì ta không dựa vào con số cụ thể nào ngoài việc `2^m · n` không quá lớn (khoảng `2^21 ≈ 2` triệu).

Ba điều cần để ý ngay:

- Ví dụ 2 nói rõ **có nhiều đáp án đúng** (`"1p3"`, `"2p2"`, `"3l1"`, ...). Vì vậy khi test, ta **không** so chuỗi kết quả với một chuỗi cố định, mà kiểm tra: (1) nó là viết tắt của `target`, (2) không là viết tắt của từ nào trong `dictionary`, (3) độ dài bằng độ dài nhỏ nhất (tính bằng brute force).
- `m <= 21` là "tín hiệu" mạnh: `2^21 ≈ 2.1` triệu — đủ nhỏ để **duyệt mọi tập con vị trí** bằng bitmask.
- Ràng buộc lạ `log2(n) + m <= 21` tương đương `n · 2^m <= 2^21`: từ điển càng lớn thì `target` càng phải ngắn. Đề "cho phép" ta làm khoảng `2^m · n` phép tính.

In [1]:
from typing import List

examples = [
    ("apple", ["blade"], "a4"),
    ("apple", ["blade", "plain", "amber"], "1p3"),   # một trong nhiều đáp án đúng
]

for target, dictionary, expected in examples:
    print(f"target = {target!r}, dictionary = {dictionary}  ->  một đáp án đúng: {expected!r}")

target = 'apple', dictionary = ['blade']  ->  một đáp án đúng: 'a4'
target = 'apple', dictionary = ['blade', 'plain', 'amber']  ->  một đáp án đúng: '1p3'


### Hiểu đề bằng tay: viết tắt và độ dài viết tắt

Hãy nghĩ về viết tắt như sau: ta đi qua `target` từng chữ; mỗi chữ hoặc **giữ lại** (keep), hoặc **bị gộp** vào một khối số. Các chữ bị gộp **liên tiếp nhau** tạo thành **một** khối, ghi bằng độ dài của khối.

```text
target :  s  u  b  s  t  i  t  u  t  i  o  n
giữ?   :  ✓  .  .  .  .  .  .  .  .  .  .  ✓      ->  "s" + "10" + "n"  = "s10n"
giữ?   :  ✓  ✓  ✓  .  .  .  .  ✓  .  .  .  .      ->  "sub" + "4" + "u" + "4" = "sub4u4"
```

Vì sao `"s55n"` sai? Nó nói "giữ `s`, rồi 5 chữ bị thay, rồi **ngay sau đó** 5 chữ nữa bị thay, rồi `n`". Hai khối số kề nhau thì người đọc không biết đó là `5` + `5` hay `55` → đề cấm. Quy tắc này có hệ quả rất đẹp: **mọi chữ bị thay liên tiếp phải gộp thành đúng một khối số**. Nhờ vậy mỗi cách chọn "giữ/không giữ" cho từng chữ ứng với **đúng một** viết tắt (ta sẽ dùng điều này ở mục 2.4).

**Độ dài viết tắt** = số chữ được giữ + số khối số. Chú ý: một khối số `10` có **2 ký tự** nhưng chỉ tính là **1**.

| Viết tắt | Chữ giữ | Khối số | Độ dài viết tắt | `len()` của chuỗi |
|---|---|---|---|---|
| `"s10n"` | 2 (`s`, `n`) | 1 (`10`) | **3** | 4 |
| `"sub4u4"` | 4 | 2 | **6** | 6 |
| `"12"` | 0 | 1 | **1** | 2 |
| `"su3i1u2on"` | 6 | 3 | **9** | 9 |
| `"substitution"` | 12 | 0 | **12** | 12 |

Cell dưới viết hàm `abbr_length_of_string` đếm độ dài viết tắt trực tiếp từ chuỗi: đi qua từng ký tự, chữ cái thì cộng 1, còn một **dãy chữ số liên tiếp** thì chỉ cộng 1 một lần.

In [2]:
def abbr_length_of_string(abbr: str) -> int:
    length = 0
    i = 0
    while i < len(abbr):
        if abbr[i].isdigit():
            while i < len(abbr) and abbr[i].isdigit():   # nuốt trọn cả khối số
                i += 1
            length += 1                                  # cả khối chỉ tính 1
        else:
            length += 1                                  # một chữ cái tính 1
            i += 1
    return length


for abbr in ["s10n", "sub4u4", "12", "su3i1u2on", "substitution"]:
    print(f"{abbr!r:16} độ dài viết tắt = {abbr_length_of_string(abbr):2}   len(chuỗi) = {len(abbr)}")

's10n'           độ dài viết tắt =  3   len(chuỗi) = 4
'sub4u4'         độ dài viết tắt =  6   len(chuỗi) = 6
'12'             độ dài viết tắt =  1   len(chuỗi) = 2
'su3i1u2on'      độ dài viết tắt =  9   len(chuỗi) = 9
'substitution'   độ dài viết tắt = 12   len(chuỗi) = 12


### "Là viết tắt của một từ trong dictionary" nghĩa là gì?

Một chuỗi viết tắt như `"4e"` có thể là viết tắt của **nhiều** từ: `"4e"` nghĩa là "4 chữ bất kỳ, rồi chữ `e`", nên nó khớp với `"apple"` **và** `"blade"`. Đề muốn viết tắt của `target` "**độc nhất**" (unique): đọc nó lên thì chỉ có thể là `target`, không thể là từ nào trong `dictionary`.

Kiểm tra lại Ví dụ 1 (`target = "apple"`, `dictionary = ["blade"]`) theo thứ tự độ dài tăng dần:

| Độ dài | Viết tắt của `"apple"` | Có phải viết tắt của `"blade"`? | Kết luận |
|---|---|---|---|
| 1 | `"5"` | Có (blade cũng 5 chữ) | loại |
| 2 | `"a4"` | Không (blade bắt đầu bằng `b`) | **nhận** |
| 2 | `"4e"` | Có (blade cũng kết thúc bằng `e`) | loại |

Không có viết tắt độ dài 1 nào dùng được, và `"a4"` dùng được ở độ dài 2 → đáp án `"a4"`. Để kiểm tra cột giữa bằng code, ta cần lời giải bài **#408** — học ngay ở mục sau.

## 2. Kiến thức nền

### 2.1. Ôn #408 Valid Word Abbreviation: kiểm tra `abbr` có phải viết tắt của `word`

Đề #408: cho `word` và `abbr`, trả về `True` nếu `abbr` là viết tắt hợp lệ của `word`. Ở #408 có thêm luật: **số không được có chữ số 0 đứng đầu** (`"a04"` sai, và `"0"` cũng sai vì không được thay chuỗi con rỗng).

Ý tưởng **hai con trỏ** (two pointers — hai biến chỉ số chạy song song trên hai chuỗi):

- `i` chạy trên `word`, `j` chạy trên `abbr`.
- Nếu `abbr[j]` là **chữ cái**: phải bằng `word[i]`, rồi cả hai cùng tiến 1.
- Nếu `abbr[j]` là **chữ số**: nếu là `'0'` → sai ngay (số có số 0 đứng đầu). Ngược lại đọc trọn cả khối số thành `num` (ví dụ `"10"` → `10`) bằng công thức `num = num * 10 + chữ_số`, rồi cho `i` **nhảy** `num` bước (bỏ qua `num` chữ của `word`).
- Kết thúc: hợp lệ ⇔ cả hai con trỏ **cùng đến cuối** (`i == len(word)` và `j == len(abbr)`).

Ví dụ mini cách đọc số: `"10"` → `num = 0*10 + 1 = 1` → `num = 1*10 + 0 = 10`.

Cell dưới viết hàm, chạy bộ test (2 ví dụ của đề #408 + vài ca của bài này), rồi dùng `trace_valid_word_abbreviation` in từng bước `i`, `j`, ký tự đang đọc và quyết định cho hai ca của Ví dụ 1: `"a4"` với `"blade"` (hỏng ngay ở chữ đầu) và `"4e"` với `"blade"` (khớp).

In [3]:
def valid_word_abbreviation(word: str, abbr: str) -> bool:
    i = j = 0
    while i < len(word) and j < len(abbr):
        if abbr[j].isdigit():
            if abbr[j] == "0":                 # số có '0' đứng đầu -> không hợp lệ
                return False
            num = 0
            while j < len(abbr) and abbr[j].isdigit():
                num = num * 10 + int(abbr[j])  # đọc trọn khối số
                j += 1
            i += num                           # nhảy qua num chữ của word
        else:
            if word[i] != abbr[j]:
                return False
            i += 1
            j += 1
    return i == len(word) and j == len(abbr)


tests_408 = [
    ("internationalization", "i12iz4n", True),    # ví dụ của đề #408
    ("apple", "a2e", False),                       # ví dụ của đề #408
    ("substitution", "s10n", True),
    ("substitution", "s010n", False),              # số 0 đứng đầu
    ("apple", "a4", True),
    ("blade", "a4", False),
    ("blade", "4e", True),
    ("apple", "6", False),                         # nhảy quá cuối từ
]
for word, abbr, expected in tests_408:
    got = valid_word_abbreviation(word, abbr)
    assert got == expected, (word, abbr, got)
    print(f"{word!r:24} {abbr!r:10} -> {got}")
print()


def trace_valid_word_abbreviation(word: str, abbr: str) -> bool:
    print(f"word = {word!r}, abbr = {abbr!r}")
    i = j = 0
    while i < len(word) and j < len(abbr):
        if abbr[j].isdigit():
            if abbr[j] == "0":
                print(f"  j={j}: gặp '0' đứng đầu -> False")
                return False
            start, num = j, 0
            while j < len(abbr) and abbr[j].isdigit():
                num = num * 10 + int(abbr[j])
                j += 1
            print(f"  j={start}: đọc số {num} -> i nhảy từ {i} tới {i + num}")
            i += num
        else:
            ok = word[i] == abbr[j]
            print(f"  i={i}, j={j}: so word[i]={word[i]!r} với abbr[j]={abbr[j]!r} -> {'khớp' if ok else 'KHÁC -> False'}")
            if not ok:
                return False
            i += 1
            j += 1
    result = i == len(word) and j == len(abbr)
    print(f"  kết thúc: i={i} (len={len(word)}), j={j} (len={len(abbr)}) -> {result}")
    return result


_ = trace_valid_word_abbreviation("blade", "a4")
_ = trace_valid_word_abbreviation("blade", "4e")

'internationalization'   'i12iz4n'  -> True
'apple'                  'a2e'      -> False
'substitution'           's10n'     -> True
'substitution'           's010n'    -> False
'apple'                  'a4'       -> True
'blade'                  'a4'       -> False
'blade'                  '4e'       -> True
'apple'                  '6'        -> False

word = 'blade', abbr = 'a4'
  i=0, j=0: so word[i]='b' với abbr[j]='a' -> KHÁC -> False
word = 'blade', abbr = '4e'
  j=0: đọc số 4 -> i nhảy từ 0 tới 4
  i=4, j=1: so word[i]='e' với abbr[j]='e' -> khớp
  kết thúc: i=5 (len=5), j=2 (len=2) -> True


### 2.2. Bitmask từ căn bản: dùng một số nguyên để ghi "tập các vị trí"

Một số nguyên trong máy tính được lưu dưới dạng các **bit** 0/1. Ví dụ `13` viết ở hệ nhị phân là `1101` (= 8 + 4 + 1). Ta đánh số bit từ **phải sang trái**, bắt đầu từ 0: bit 0 của 13 là 1, bit 1 là 0, bit 2 là 1, bit 3 là 1.

**Bitmask** (mặt nạ bit) là dùng một số nguyên để biểu diễn một **tập con** của `{0, 1, ..., m-1}`: bit `i` bằng 1 ⇔ phần tử `i` nằm trong tập. Với bài này, phần tử là **vị trí chữ** trong `target`:

> **Quy ước của notebook:** bit `i` của `mask` bằng 1 ⇔ **giữ chữ** ở vị trí `i` của `target`; bằng 0 ⇔ chữ đó bị gộp vào khối số.

Các phép toán bit cần biết (mỗi phép đều có ví dụ ở cell dưới):

| Phép | Đọc là | Ý nghĩa với tập vị trí |
|---|---|---|
| `1 << i` | dịch 1 sang trái `i` bước | tập chỉ có vị trí `i` (số `2^i`) |
| `mask >> i & 1` | lấy bit `i` | vị trí `i` có trong tập không? (1/0) |
| `mask \| (1 << i)` | OR | thêm vị trí `i` vào tập |
| `a & b` | AND | **giao** hai tập; `a & b != 0` ⇔ hai tập có phần tử chung |
| `~a & full` | NOT rồi cắt về `m` bit | **phần bù** trong `{0..m-1}`, với `full = (1 << m) - 1` |
| `a.bit_count()` | đếm bit 1 | **số phần tử** của tập (Python ≥ 3.10; bản cũ dùng `bin(a).count("1")`) |

Một bẫy rất hay gặp: `bin(13)` in `'0b1101'`, tức in **bit cao trước** (bit 3 bên trái). Nhưng chuỗi `target` lại đọc **vị trí 0 trước**. Để nhìn mask "thẳng hàng" với chữ, notebook dùng hàm `bits(mask, m)` in **vị trí 0 bên trái**.

In [4]:
def bits(mask: int, m: int) -> str:
    # In bit theo thứ tự VỊ TRÍ: ký tự thứ i của chuỗi = bit i của mask
    return "".join("1" if mask >> i & 1 else "0" for i in range(m))


m = 5
mask = 0
mask = mask | (1 << 0)          # giữ vị trí 0
mask = mask | (1 << 3)          # giữ vị trí 3
print("mask =", mask, "| bin(mask) =", bin(mask), "| bits(mask, 5) =", bits(mask, m))
print("vị trí 3 có được giữ?", mask >> 3 & 1, "| vị trí 1?", mask >> 1 & 1)
print("số chữ được giữ (bit_count):", mask.bit_count())

full = (1 << m) - 1
print("full =", full, "->", bits(full, m))
print("~mask (chưa cắt) =", ~mask, " <- số ÂM! Python không có giới hạn số bit")
print("~mask & full =", ~mask & full, "->", bits(~mask & full, m), "(các vị trí bị gộp)")

a, b = 0b00110, 0b01100        # literal 0b... viết bit CAO trước; bits() in theo VỊ TRÍ nên trông "ngược"
print("a =", bits(a, m), " b =", bits(b, m), " a & b =", bits(a & b, m), "-> có chung vị trí?", (a & b) != 0)

mask = 9 | bin(mask) = 0b1001 | bits(mask, 5) = 10010
vị trí 3 có được giữ? 1 | vị trí 1? 0
số chữ được giữ (bit_count): 2
full = 31 -> 11111
~mask (chưa cắt) = -10  <- số ÂM! Python không có giới hạn số bit
~mask & full = 22 -> 01101 (các vị trí bị gộp)
a = 01100  b = 00110  a & b = 00100 -> có chung vị trí? True


### 2.3. Duyệt **mọi** tập con bằng `range(1 << m)`

Với `m` vị trí, mỗi vị trí có 2 lựa chọn (giữ / không giữ) → có `2^m` tập con. Thật tiện: các số nguyên `0, 1, 2, ..., 2^m - 1` chính là **tất cả** các mask `m` bit, mỗi mask đúng một lần. Vậy chỉ cần `for mask in range(1 << m)` là đã duyệt hết mọi tập con, không cần đệ quy.

Ví dụ `m = 3`: có `2^3 = 8` mask, từ `000` (không giữ gì) tới `111` (giữ hết).

In [5]:
m = 3
word = "cat"
for mask in range(1 << m):
    kept = [i for i in range(m) if mask >> i & 1]
    print(f"mask={mask}  bits={bits(mask, m)}  vị trí giữ={kept}  chữ giữ={[word[i] for i in kept]}")
print("tổng số mask:", 1 << m)

mask=0  bits=000  vị trí giữ=[]  chữ giữ=[]
mask=1  bits=100  vị trí giữ=[0]  chữ giữ=['c']
mask=2  bits=010  vị trí giữ=[1]  chữ giữ=['a']
mask=3  bits=110  vị trí giữ=[0, 1]  chữ giữ=['c', 'a']
mask=4  bits=001  vị trí giữ=[2]  chữ giữ=['t']
mask=5  bits=101  vị trí giữ=[0, 2]  chữ giữ=['c', 't']
mask=6  bits=011  vị trí giữ=[1, 2]  chữ giữ=['a', 't']
mask=7  bits=111  vị trí giữ=[0, 1, 2]  chữ giữ=['c', 'a', 't']
tổng số mask: 8


### 2.4. Ôn #320 Generalized Abbreviation: từ mask sang chuỗi viết tắt

Đề #320: liệt kê **mọi** viết tắt của một từ. Ví dụ `"word"` có 16 viết tắt:
`"4","3d","2r1","2rd","1o2","1o1d","1or1","1ord","w3","w2d","w1r1","w1rd","wo2","wo1d","wor1","word"`.

Với bitmask, cách làm rất máy móc. Hàm `mask_to_abbr(target, mask)` đi qua từng vị trí:

- Bit `i` = 0 → chữ này bị gộp: tăng biến đếm `cnt`.
- Bit `i` = 1 → giữ chữ: nếu `cnt > 0` thì **xả** khối số `str(cnt)` ra trước, đặt `cnt = 0`, rồi ghi chữ.
- Hết chuỗi mà `cnt > 0` → xả khối số cuối.

Vì các bit 0 **liên tiếp** luôn được đếm chung vào **một** `cnt`, ta không bao giờ tạo ra hai khối số kề nhau → viết tắt luôn hợp lệ. Ngược lại, mỗi viết tắt hợp lệ cho biết chính xác chữ nào được giữ → **mask ↔ viết tắt là tương ứng 1–1**. Vì thế một từ dài `m` có **đúng `2^m`** viết tắt.

Cell dưới còn kiểm chứng tương ứng 1–1 trên `"apple"`: `2^5 = 32` mask cho ra 32 viết tắt **khác nhau**, mỗi cái đều được hàm #408 xác nhận là viết tắt của `"apple"`; và đếm số viết tắt ở mỗi độ dài — độ dài nhỏ nhất là 1 (chỉ có `"5"`), lớn nhất là 5.

In [6]:
def mask_to_abbr(target: str, mask: int) -> str:
    out = []
    cnt = 0
    for i, ch in enumerate(target):
        if mask >> i & 1:            # giữ chữ i
            if cnt:
                out.append(str(cnt)) # xả khối số đang đếm
                cnt = 0
            out.append(ch)
        else:                        # chữ i bị gộp vào khối số
            cnt += 1
    if cnt:
        out.append(str(cnt))
    return "".join(out)


def all_abbreviations(word: str) -> List[str]:
    return [mask_to_abbr(word, mask) for mask in range(1 << len(word))]


print(mask_to_abbr("apple", 0b00001), mask_to_abbr("apple", 0b10000), mask_to_abbr("apple", 0b00010))
abbrs = all_abbreviations("word")
print(len(abbrs), abbrs)
expected_320 = ["4","3d","2r1","2rd","1o2","1o1d","1or1","1ord","w3","w2d","w1r1","w1rd","wo2","wo1d","wor1","word"]
assert sorted(abbrs) == sorted(expected_320)
print("Khớp đúng 16 viết tắt của đề #320")


from collections import Counter

abbrs = all_abbreviations("apple")
assert len(set(abbrs)) == 32
assert all(valid_word_abbreviation("apple", a) for a in abbrs)
by_len = Counter(abbr_length_of_string(a) for a in abbrs)
print("số viết tắt khác nhau:", len(set(abbrs)))
print("số viết tắt theo độ dài:", dict(sorted(by_len.items())))
print("độ dài 2:", [a for a in abbrs if abbr_length_of_string(a) == 2])

a4 4e 1p3
16 ['4', 'w3', '1o2', 'wo2', '2r1', 'w1r1', '1or1', 'wor1', '3d', 'w2d', '1o1d', 'wo1d', '2rd', 'w1rd', '1ord', 'word']
Khớp đúng 16 viết tắt của đề #320
số viết tắt khác nhau: 32
số viết tắt theo độ dài: {1: 1, 2: 2, 3: 6, 4: 10, 5: 13}
độ dài 2: ['a4', '4e']


## 3. Cách 1 — Brute force: sinh hết, sắp theo độ dài, kiểm tra từng cái

Đây là cách người mới tự nghĩ ra ngay sau khi biết #320 và #408:

1. Sinh cả `2^m` viết tắt của `target` (mục 2.4).
2. Sắp xếp chúng theo **độ dài viết tắt** tăng dần.
3. Đi lần lượt: viết tắt nào **không** là viết tắt của từ nào trong `dictionary` (kiểm tra bằng hàm #408) thì trả về luôn — vì ta đi theo độ dài tăng dần, cái đầu tiên tìm thấy chắc chắn ngắn nhất.

Đáp án luôn tồn tại: viết tắt "giữ hết chữ" chính là `target`, và `dictionary` không chứa `target`, nên chuỗi `target` không phải viết tắt của từ nào khác.

Python sắp xếp **ổn định** (stable sort): các viết tắt cùng độ dài giữ nguyên thứ tự mask tăng dần. Vì vậy kết quả của Cách 1 là **xác định** — dùng làm trọng tài rất tiện.

In [7]:
def min_abbr_bruteforce(target: str, dictionary: List[str]) -> str:
    candidates = sorted(all_abbreviations(target), key=abbr_length_of_string)
    for abbr in candidates:
        if not any(valid_word_abbreviation(word, abbr) for word in dictionary):
            return abbr
    raise ValueError("không thể xảy ra nếu dictionary không chứa target")


for target, dictionary, expected in examples:
    got = min_abbr_bruteforce(target, dictionary)
    print(f"{target!r} {dictionary} -> {got!r} (độ dài {abbr_length_of_string(got)}), đề gợi ý {expected!r}")
print(min_abbr_bruteforce("apple", []))        # dictionary rỗng -> "5"

'apple' ['blade'] -> 'a4' (độ dài 2), đề gợi ý 'a4'
'apple' ['blade', 'plain', 'amber'] -> '1p3' (độ dài 3), đề gợi ý '1p3'
5


### Cách 1 vướng ở đâu?

Đếm công việc: có `2^m` viết tắt; mỗi viết tắt tốn `O(m)` để **dựng chuỗi**, rồi bị đem so với **mọi** từ trong từ điển, mỗi lần so bằng hai con trỏ tốn tới `O(m)`. Tệ nhất khoảng `2^m · n · m` thao tác **trên chuỗi** (chưa kể chi phí sắp xếp `2^m` chuỗi).

Với `m = 21`, `n = 1`: `2^21 · 21 ≈ 44` triệu thao tác chuỗi — Python sẽ mất vài chục giây. Ba chỗ lãng phí rõ ràng:

1. **So với từ khác độ dài**: một viết tắt của `target` luôn "mô tả" đúng `m` chữ, nên **không bao giờ** khớp với từ có độ dài khác `m`. Vậy mà Cách 1 vẫn đem đi so.
2. **Dựng chuỗi rồi phân tích lại chuỗi**: ta tạo `"1p3"` từ mask, rồi hàm #408 lại phải đọc `"1p3"` để hiểu "vị trí 1 là `p`". Thông tin đó **đã có sẵn** trong mask!
3. **So từng ký tự** mỗi lần: điều kiện "khớp" thật ra có thể gói vào **một phép `&`** giữa hai số nguyên (mục 4).

Cell dưới đếm số lần gọi hàm #408 của Cách 1 trong một ca **tệ nhất**: `target = "aaa...a"`, và với **mỗi** vị trí `i` từ điển có một từ chỉ khác `target` đúng ở vị trí `i` (ví dụ `m = 4`: `"baaa"`, `"abaa"`, `"aaba"`, `"aaab"`), cộng thêm 2 từ khác độ dài. Khi đó **mọi** vị trí đều phải giữ → đáp án là chính `target` (độ dài `m`, đứng cuối danh sách đã sắp) → Cách 1 phải đi gần hết `2^m` viết tắt.

In [8]:
def count_checks_bruteforce(target: str, dictionary: List[str]) -> int:
    checks = 0
    for abbr in sorted(all_abbreviations(target), key=abbr_length_of_string):
        hit = False
        for word in dictionary:
            checks += 1
            if valid_word_abbreviation(word, abbr):
                hit = True
                break
        if not hit:
            return checks
    return checks


for m in [4, 6, 8, 10]:
    target = "a" * m
    dictionary = ["a" * i + "b" + "a" * (m - 1 - i) for i in range(m)] + ["a" * (m + 1), "b" * (m - 1)]
    assert min_abbr_bruteforce(target, dictionary) == target
    print(f"m = {m:2}: n = {len(dictionary):2}, số viết tắt = {2**m:5}, số lần gọi #408 = {count_checks_bruteforce(target, dictionary)}")

m =  4: n =  6, số viết tắt =    16, số lần gọi #408 = 32
m =  6: n =  8, số viết tắt =    64, số lần gọi #408 = 128
m =  8: n = 10, số viết tắt =   256, số lần gọi #408 = 512
m = 10: n = 12, số viết tắt =  1024, số lần gọi #408 = 2048


## 4. Bước đột phá: biến "khớp viết tắt" thành phép `&` trên bitmask

### 4.1. Chỉ từ **cùng độ dài** với `target` mới đáng quan tâm

Mọi viết tắt của `target` có tổng (số chữ giữ + các số trong khối) đúng bằng `m`. Hàm #408 chỉ trả `True` khi con trỏ `i` đi **vừa khít** hết từ. Nên với từ có độ dài khác `m`, kết quả **luôn** `False`. Ví dụ: `"a4"` "mô tả" 5 chữ, không thể khớp `"applepie"` (8 chữ) hay `"app"` (3 chữ).

→ Bước đầu tiên: **lọc** `dictionary`, chỉ giữ từ có `len(word) == m`.

In [9]:
for abbr in all_abbreviations("apple"):
    for word in ["applepie", "app", "apples"]:
        assert not valid_word_abbreviation(word, abbr)
print("Không viết tắt nào của 'apple' khớp từ khác độ dài -> có thể lọc bỏ các từ đó.")

dictionary = ["blade", "applepie", "plain", "app", "amber"]
same_len = [w for w in dictionary if len(w) == len("apple")]
print("sau khi lọc:", same_len)

Không viết tắt nào của 'apple' khớp từ khác độ dài -> có thể lọc bỏ các từ đó.
sau khi lọc: ['blade', 'plain', 'amber']


### 4.2. Viết tắt (mask `K`) khớp từ `w` ⇔ `w` **giống** `target` tại **mọi vị trí được giữ**

Một viết tắt nói: "ở các vị trí giữ, chữ phải là thế này; các vị trí còn lại là gì cũng được". Ví dụ `"1p3"` (mask giữ vị trí 1) chỉ đòi hỏi `w[1] == 'p'`. Vậy với `w` cùng độ dài:

> Mask `K` khớp `w`  ⇔  với mọi `i` mà bit `i` của `K` bằng 1: `w[i] == target[i]`.

Ta sẽ kiểm chứng phát biểu này bằng code (so với hàm #408 trên **mọi** mask của `"apple"`) ngay sau khi gói nó gọn hơn.

### 4.3. Diff-mask: gói cả điều kiện vào một phép `&`

Với mỗi từ `w` cùng độ dài, định nghĩa **diff-mask** `D(w)`: bit `i` = 1 ⇔ `w[i] != target[i]` (vị trí hai từ **khác** nhau).

Từ 4.2: `K` khớp `w` ⇔ `K` **không** giữ vị trí nào mà `w` khác `target` ⇔ `K & D(w) == 0`.

Vậy:

> **Viết tắt `K` là hợp lệ (không khớp từ nào)  ⇔  `K & D(w) != 0` với MỌI từ `w` cùng độ dài.**

Hình dung: mỗi từ trong từ điển là một "ổ khóa" có vài lỗ (các vị trí khác `target`); viết tắt của ta phải **cắm vào ít nhất một lỗ** của **mọi** ổ khóa — tức là giữ lại ít nhất một chữ "phân biệt được" `target` với từ đó. (Trong toán, đây là bài **hitting set**: chọn tập điểm chạm vào mọi tập cho trước.)

Ví dụ 2, `target = "apple"` (in theo vị trí, trái = vị trí 0):

| Từ | So từng chữ với `apple` | Diff-mask `bits` |
|---|---|---|
| `blade` | b≠a, l≠p, a≠p, d≠l, e=e | `11110` |
| `plain` | p≠a, l≠p, a≠p, i≠l, n≠e | `11111` |
| `amber` | a=a, m≠p, b≠p, e≠l, r≠e | `01111` |

- `"a4"` = mask `10000`: `& amber(01111) = 0` → khớp `amber` → loại (đúng như đề nói).
- `"4e"` = mask `00001`: `& blade(11110) = 0` → khớp `blade` → loại.
- `"1p3"` = mask `01000`: chạm cả ba (`blade`, `plain`, `amber` đều có bit 1 ở vị trí 1) → **hợp lệ**.

Cell dưới: (1) kiểm chứng 4.2, (2) tính diff-mask cho Ví dụ 2 và thử 4 viết tắt ngắn.

In [10]:
def matches_by_positions(target: str, word: str, mask: int) -> bool:
    # chỉ dùng khi len(word) == len(target)
    return all(word[i] == target[i] for i in range(len(target)) if mask >> i & 1)


target = "apple"
for word in ["blade", "plain", "amber", "apply", "bpple"]:
    for mask in range(1 << len(target)):
        assert matches_by_positions(target, word, mask) == valid_word_abbreviation(word, mask_to_abbr(target, mask))
print("Hai cách kiểm tra cho kết quả giống hệt nhau trên mọi mask.")
print("'1p3' khớp 'amber'?", matches_by_positions("apple", "amber", 0b00010), " khớp 'spill'?", matches_by_positions("apple", "spill", 0b00010))


def diff_mask(target: str, word: str) -> int:
    d = 0
    for i in range(len(target)):
        if word[i] != target[i]:
            d |= 1 << i
    return d


target, dictionary = "apple", ["blade", "plain", "amber"]
diffs = [diff_mask(target, w) for w in dictionary if len(w) == len(target)]
for w, d in zip(dictionary, diffs):
    print(f"{w}: diff = {bits(d, 5)}  (số nguyên {d})")

for abbr_mask in [0b00000, 0b00001, 0b10000, 0b00010]:
    hits = [bool(abbr_mask & d) for d in diffs]
    print(f"{mask_to_abbr(target, abbr_mask):4} mask={bits(abbr_mask, 5)} chạm từng từ = {hits} -> hợp lệ? {all(hits)}")

Hai cách kiểm tra cho kết quả giống hệt nhau trên mọi mask.
'1p3' khớp 'amber'? False  khớp 'spill'? True
blade: diff = 11110  (số nguyên 15)
plain: diff = 11111  (số nguyên 31)
amber: diff = 01111  (số nguyên 30)
5    mask=00000 chạm từng từ = [False, False, False] -> hợp lệ? False
a4   mask=10000 chạm từng từ = [True, True, False] -> hợp lệ? False
4e   mask=00001 chạm từng từ = [False, True, True] -> hợp lệ? False
1p3  mask=01000 chạm từng từ = [True, True, True] -> hợp lệ? True


### 4.4. Tính độ dài viết tắt **trực tiếp từ mask** (không dựng chuỗi)

Độ dài = (số chữ giữ) + (số khối số). Số chữ giữ là `mask.bit_count()`. Số khối số = số **đoạn bit 0 liên tiếp** (gọi là các "run" 0) trong `m` bit.

Cách đếm số run 0 bằng phép bit:

1. `zeros = ~mask & full` — tập vị trí bị gộp (nhớ `& full` để khỏi bị số âm, mục 2.2).
2. Một run bắt đầu tại vị trí `i` ⇔ `i` thuộc `zeros` **và** `i - 1` không thuộc `zeros` (hoặc `i = 0`).
3. `zeros << 1` dời mọi bit lên một vị trí: bit `i` của `zeros << 1` chính là bit `i - 1` của `zeros`. Vậy `starts = zeros & ~(zeros << 1)` giữ đúng các vị trí **bắt đầu run**.
4. Độ dài = `mask.bit_count() + starts.bit_count()`.

Ví dụ `"1p3"` (`m = 5`, mask = `01000` theo vị trí):

```text
vị trí          :  0 1 2 3 4
mask            :  0 1 0 0 0      -> giữ 1 chữ
zeros           :  1 0 1 1 1
zeros << 1      :  0 1 0 1 1      (mỗi bit dời sang phải 1 ô trong cách in theo vị trí)
~(zeros << 1)   :  1 0 1 0 0
starts          :  1 0 1 0 0      -> 2 run 0 (bắt đầu ở vị trí 0 và 2)
độ dài          :  1 + 2 = 3       ✓ ("1p3")
```

(`zeros << 1` có thể sinh thêm bit thứ `m`, nhưng `zeros` không có bit đó nên phép `&` tự loại bỏ.)

In [11]:
def abbr_len_slow(target_len: int, mask: int) -> int:
    # cách chậm: dựng chuỗi rồi đếm
    return abbr_length_of_string(mask_to_abbr("x" * target_len, mask))


def abbr_len_fast(mask: int, m: int) -> int:
    full = (1 << m) - 1
    zeros = ~mask & full
    starts = zeros & ~(zeros << 1)
    return mask.bit_count() + starts.bit_count()


mask, m = 0b00010, 5
zeros = ~mask & ((1 << m) - 1)
print("mask   =", bits(mask, m))
print("zeros  =", bits(zeros, m))
print("starts =", bits(zeros & ~(zeros << 1), m))
print("độ dài =", abbr_len_fast(mask, m), "| chuỗi:", mask_to_abbr("apple", mask))

for m in range(1, 11):
    for mask in range(1 << m):
        assert abbr_len_fast(mask, m) == abbr_len_slow(m, mask)
print("abbr_len_fast khớp cách chậm trên mọi mask với m = 1..10")

mask   = 01000
zeros  = 10111
starts = 10100
độ dài = 3 | chuỗi: 1p3
abbr_len_fast khớp cách chậm trên mọi mask với m = 1..10


## 5. Cách 2 — Diff-mask + duyệt mọi mask (cách chính)

### Cách 1 vướng ở đâu → mấu chốt cải tiến

Cách 1 làm việc trên **chuỗi**: dựng chuỗi, sắp chuỗi, phân tích chuỗi, so với cả từ khác độ dài. Mục 4 cho ta ba công cụ để chuyển hết sang **số nguyên**:

- Lọc chỉ giữ từ cùng độ dài, tính trước mỗi từ một `diff`.
- Kiểm tra hợp lệ: `all(mask & d for d in diffs)` — mỗi từ chỉ một phép `&`.
- Độ dài: `abbr_len_fast(mask, m)` — vài phép bit, không dựng chuỗi.

### Thuật toán

1. `diffs` = diff-mask của mọi từ cùng độ dài.
2. `best_len = m + 1` (lớn hơn mọi độ dài có thể), `best_mask = full`.
3. Với mỗi `mask` trong `range(1 << m)`:
   - tính `length = abbr_len_fast(mask, m)`;
   - **chỉ khi** `length < best_len` mới tốn công kiểm tra `all(mask & d for d in diffs)` (phép so độ dài rẻ, làm trước để loại nhanh);
   - nếu hợp lệ → cập nhật `best_len, best_mask`.
4. Dựng chuỗi **một lần duy nhất**: `mask_to_abbr(target, best_mask)`.

Vì sao `best_mask = full` là an toàn? Mask "giữ hết" chạm mọi `diff` khác 0, mà mọi từ trong `dictionary` đều khác `target` nên `diff != 0`. Và `full` có độ dài `m < m + 1` nên chắc chắn được xét — đáp án luôn tồn tại.

Nếu không có từ cùng độ dài thì `diffs` rỗng, `all([])` là `True`, nên ngay `mask = 0` (độ dài 1, chuỗi `str(m)`) đã thắng — đúng: `"5"` là đáp án khi `dictionary = []`.

In [12]:
def min_abbr_diffmask(target: str, dictionary: List[str]) -> str:
    m = len(target)
    diffs = [diff_mask(target, w) for w in dictionary if len(w) == m]
    full = (1 << m) - 1
    best_len, best_mask = m + 1, full
    for mask in range(1 << m):
        length = abbr_len_fast(mask, m)
        if length < best_len and all(mask & d for d in diffs):
            best_len, best_mask = length, mask
    return mask_to_abbr(target, best_mask)


for target, dictionary, expected in examples:
    got = min_abbr_diffmask(target, dictionary)
    print(f"{target!r} {dictionary} -> {got!r} (độ dài {abbr_length_of_string(got)})")
print(min_abbr_diffmask("apple", []), min_abbr_diffmask("apple", ["applepie", "app"]))

'apple' ['blade'] -> 'a4' (độ dài 2)
'apple' ['blade', 'plain', 'amber'] -> '1p3' (độ dài 3)
5 5


### Trace Cách 2 trên Ví dụ 2

Hàm `trace_diffmask` in từng mask **có được kiểm tra** (tức `length < best_len`), từ nào nó **không chạm** (tức viết tắt đó khớp từ ấy), và lúc `best` được cập nhật. Để output gọn, trace dừng sau khi `best_len` đạt cận dưới thực tế thì chỉ đếm, không in nữa.

In [13]:
def trace_diffmask(target: str, dictionary: List[str], max_lines: int = 12):
    m = len(target)
    words = [w for w in dictionary if len(w) == m]
    diffs = [diff_mask(target, w) for w in words]
    best_len, best_mask = m + 1, (1 << m) - 1
    printed = checked = 0
    print(f"diffs: " + ", ".join(f"{w}={bits(d, m)}" for w, d in zip(words, diffs)))
    for mask in range(1 << m):
        length = abbr_len_fast(mask, m)
        if length >= best_len:
            continue                                    # bị loại chỉ bằng phép so độ dài
        checked += 1
        blocked_by = [w for w, d in zip(words, diffs) if mask & d == 0]
        if not blocked_by:
            best_len, best_mask = length, mask
        if printed < max_lines:
            note = "HỢP LỆ -> cập nhật best" if not blocked_by else f"khớp {blocked_by} -> loại"
            print(f"mask={bits(mask, m)} {mask_to_abbr(target, mask):6} len={length} best_len={best_len}  {note}")
            printed += 1
    print(f"Số mask phải kiểm tra all(): {checked} / {1 << m}")
    print("Kết quả:", mask_to_abbr(target, best_mask))


trace_diffmask("apple", ["blade", "plain", "amber"])

diffs: blade=11110, plain=11111, amber=01111
mask=00000 5      len=1 best_len=6  khớp ['blade', 'plain', 'amber'] -> loại
mask=10000 a4     len=2 best_len=6  khớp ['amber'] -> loại
mask=01000 1p3    len=3 best_len=3  HỢP LỆ -> cập nhật best
mask=00001 4e     len=2 best_len=3  khớp ['blade'] -> loại
Số mask phải kiểm tra all(): 4 / 32
Kết quả: 1p3


Đọc trace bằng bảng (khớp với output ở trên):

| mask (theo vị trí) | Viết tắt | Độ dài | `best_len` trước | Không chạm (khớp) từ | Quyết định |
|---|---|---|---|---|---|
| `00000` | `5` | 1 | 6 | blade, plain, amber | loại |
| `10000` | `a4` | 2 | 6 | amber | loại |
| `01000` | `1p3` | 3 | 6 | — | **cập nhật** `best_len = 3` |
| `00001` | `4e` | 2 | 3 | blade | loại |
| 28 mask còn lại | | ≥ 3 | 3 | (không cần xét) | bỏ qua chỉ bằng phép so độ dài |

Ý nghĩa: mask `3` (`11000`, `"ap3"`) và các mask khác đứng giữa có độ dài ≥ 3 nên **không** được in. Ngay sau khi tìm được `"1p3"` (độ dài 3), mọi mask có độ dài ≥ 3 bị loại chỉ bằng **một phép so sánh số nguyên**. Chỉ còn mask độ dài 1–2 — trong cả 32 mask chỉ có 3 cái như vậy (`"5"`, `"a4"`, `"4e"`) — mới bị kiểm tra `all(...)`, và chúng đều hỏng. Tổng cộng chỉ **4 / 32** mask phải kiểm tra `all(...)`. Kết quả `"1p3"` trùng với đáp án mẫu của đề.

### 5.1. Biến thể của Cách 2: duyệt mask theo độ dài **tăng dần**, gặp cái hợp lệ đầu tiên là dừng

Thay vì duyệt mask theo thứ tự `0, 1, 2, ...` rồi giữ min, ta có thể **sắp** các mask theo độ dài viết tắt rồi dừng ở mask hợp lệ đầu tiên — giống tinh thần Cách 1 nhưng làm trên số nguyên.

- Ưu điểm: dừng rất sớm khi đáp án ngắn (thường gặp).
- Nhược điểm: phải tạo và sắp cả `2^m` số (`O(2^m log 2^m)` thời gian, `O(2^m)` bộ nhớ — với `m = 21` là ~2 triệu số).

Trace dưới in từng mask theo thứ tự xét, cùng từ chặn nó.

In [14]:
def min_abbr_by_length(target: str, dictionary: List[str]) -> str:
    m = len(target)
    diffs = [diff_mask(target, w) for w in dictionary if len(w) == m]
    order = sorted(range(1 << m), key=lambda mask: abbr_len_fast(mask, m))   # sort ổn định
    for mask in order:
        if all(mask & d for d in diffs):
            return mask_to_abbr(target, mask)


def trace_by_length(target: str, dictionary: List[str]):
    m = len(target)
    words = [w for w in dictionary if len(w) == m]
    diffs = [diff_mask(target, w) for w in words]
    for step, mask in enumerate(sorted(range(1 << m), key=lambda k: abbr_len_fast(k, m)), 1):
        blocked = [w for w, d in zip(words, diffs) if mask & d == 0]
        print(f"bước {step}: {mask_to_abbr(target, mask):5} len={abbr_len_fast(mask, m)}  khớp {blocked if blocked else 'không từ nào -> TRẢ VỀ'}")
        if not blocked:
            return mask_to_abbr(target, mask)


print(min_abbr_by_length("apple", ["blade"]), min_abbr_by_length("apple", ["blade", "plain", "amber"]))
_ = trace_by_length("apple", ["blade", "plain", "amber"])

a4 1p3
bước 1: 5     len=1  khớp ['blade', 'plain', 'amber']
bước 2: a4    len=2  khớp ['amber']
bước 3: 4e    len=2  khớp ['blade']
bước 4: 1p3   len=3  khớp không từ nào -> TRẢ VỀ


### Độ phức tạp Cách 2

Đặt `m = len(target)`, `n = len(dictionary)`, `L` = độ dài tối đa của một từ trong từ điển, `n'` = số từ **cùng độ dài** với `target` (`n' <= n`).

- **Tiền xử lý:** lọc theo độ dài `O(n)` (Python lưu sẵn độ dài chuỗi, `len()` là `O(1)`), tính `diff` cho `n'` từ: `O(n' · m)`.
- **Vòng chính:** `2^m` mask; mỗi mask tốn `O(1)` phép bit cho độ dài (coi `bit_count` trên số ≤ 21 bit là `O(1)`), cộng tối đa `O(n')` phép `&` khi phải kiểm tra. Tệ nhất `O(2^m · n')`.
- Với ràng buộc `log2(n) + m <= 21` ⇒ `2^m · n <= 2^21 ≈ 2.1` triệu → chạy được trong Python (khoảng vài trăm mili-giây cho trường hợp `m = 21`).
- **Bộ nhớ:** `O(n')` cho `diffs` (biến thể 5.1 tốn thêm `O(2^m)` cho danh sách đã sắp).

## 6. Cách 3 — DFS + cắt tỉa (branch and bound)

### Cách 2 còn vướng ở đâu?

Cách 2 **luôn** đi hết `2^m` mask, kể cả khi đáp án rất ngắn và đã tìm ra từ sớm. Ví dụ `m = 21`, một từ khác `target` ở mọi vị trí: `"a20"` (độ dài 2) hợp lệ, nhưng vẫn phải lặp 2 triệu lần để "chắc chắn". Ta muốn: **dựng viết tắt từng bước** và **bỏ cả một nhánh** ngay khi biết nó không thể tốt hơn.

### Kiến thức nền: đệ quy (recursion) và DFS

**Đệ quy** là hàm tự gọi lại chính nó trên một bài toán nhỏ hơn, có **điểm dừng** (base case). **DFS** (Depth-First Search — tìm kiếm theo chiều sâu) là cách đi một nhánh lựa chọn tới tận cùng rồi mới quay lại thử nhánh khác; đệ quy là cách viết DFS tự nhiên nhất.

Sinh viết tắt bằng DFS: đứng ở vị trí `i`, **ngay trước `i` là một chữ được giữ (hoặc đầu chuỗi)**, ta có các lựa chọn:

- **Giữ** chữ `i` → đi tiếp tới `i + 1`.
- **Gộp** khối chữ `i .. j-1` thành số `j - i` (với `j` từ `i + 1` tới `m`), rồi:
  - nếu `j == m`: hết chuỗi;
  - nếu `j < m`: **bắt buộc giữ** chữ `j` (để khối số sau không kề khối này) → đi tiếp tới `j + 1`.

Nhờ "sau một khối số bắt buộc là một chữ", DFS **không bao giờ** sinh hai khối số kề nhau. Kiểm chứng: DFS trên `"word"` ra đúng 16 viết tắt của #320.

In [15]:
def abbreviations_dfs(word: str) -> List[str]:
    m = len(word)
    result = []

    def dfs(i, parts):
        if i == m:                                   # base case: đã đi hết từ
            result.append("".join(parts))
            return
        dfs(i + 1, parts + [word[i]])                # lựa chọn 1: giữ chữ i
        for j in range(i + 1, m + 1):                # lựa chọn 2: gộp word[i:j] thành số
            if j == m:
                dfs(m, parts + [str(j - i)])
            else:
                dfs(j + 1, parts + [str(j - i), word[j]])   # sau khối số bắt buộc giữ chữ j

    dfs(0, [])
    return result


res = abbreviations_dfs("word")
print(len(res), res)
assert sorted(res) == sorted(all_abbreviations("word"))
assert sorted(abbreviations_dfs("apple")) == sorted(all_abbreviations("apple"))
print("DFS sinh đúng tập viết tắt như bitmask (#320)")

16 ['word', 'wor1', 'wo1d', 'wo2', 'w1rd', 'w1r1', 'w2d', 'w3', '1ord', '1or1', '1o1d', '1o2', '2rd', '2r1', '3d', '4']
DFS sinh đúng tập viết tắt như bitmask (#320)


### Thêm "branch and bound": hai luật cắt tỉa

Thay vì `parts` (danh sách chuỗi), DFS giờ mang theo `mask` (các vị trí đã giữ) và `length` (độ dài viết tắt **đã dựng** — mỗi chữ giữ +1, mỗi khối số +1). `length` chỉ **tăng** khi đi sâu, nên:

- **Luật 1 — cận (bound):** nếu `length >= best_len` thì nhánh này không thể cho đáp án **ngắn hơn** → `return` ngay.
- **Luật 2 — nhánh chết:** khi đứng ở vị trí `i`, các vị trí `< i` đã được quyết định xong. Nếu có một `diff` mà **mọi bit của nó đều `< i`** và `mask & diff == 0`, thì không còn cách nào chạm từ đó nữa → nhánh chắc chắn hỏng → `return`. Điều kiện "mọi bit `< i`" viết là `diff & low == diff` với `low = (1 << i) - 1`.

Tới `i == m` (lá): nếu `mask` chạm mọi `diff` → cập nhật `best`.

Thứ tự thử: **giữ chữ trước**, rồi gộp khối ngắn → dài. Thứ tự này không ảnh hưởng tính đúng, chỉ ảnh hưởng việc tìm sớm một đáp án tốt để Luật 1 cắt nhiều hơn.

In [16]:
def min_abbr_dfs(target: str, dictionary: List[str], stats: dict = None) -> str:
    m = len(target)
    diffs = [diff_mask(target, w) for w in dictionary if len(w) == m]
    best = [m + 1, (1 << m) - 1]          # [best_len, best_mask] — list để hàm lồng sửa được
    visited = [0]

    def dfs(i, mask, length):
        visited[0] += 1
        if length >= best[0]:                              # Luật 1: không thể ngắn hơn
            return
        if i == m:                                         # lá
            if all(mask & d for d in diffs):
                best[0], best[1] = length, mask
            return
        low = (1 << i) - 1
        for d in diffs:                                    # Luật 2: có từ không còn cách chạm
            if d & low == d and mask & d == 0:
                return
        dfs(i + 1, mask | (1 << i), length + 1)            # giữ chữ i
        for j in range(i + 1, m + 1):                      # gộp target[i:j]
            if j == m:
                dfs(m, mask, length + 1)
            else:
                dfs(j + 1, mask | (1 << j), length + 2)    # khối số + chữ j

    dfs(0, 0, 0)
    if stats is not None:
        stats["visited"] = visited[0]
    return mask_to_abbr(target, best[1])


for target, dictionary, expected in examples:
    st = {}
    got = min_abbr_dfs(target, dictionary, st)
    print(f"{target!r} {dictionary} -> {got!r} (độ dài {abbr_length_of_string(got)}), số nút DFS = {st['visited']}")

'apple' ['blade'] -> 'a4' (độ dài 2), số nút DFS = 21
'apple' ['blade', 'plain', 'amber'] -> 'ap3' (độ dài 3), số nút DFS = 30


Chú ý Ví dụ 2: Cách 3 trả `"ap3"` chứ không phải `"1p3"` — cũng **độ dài 3** và cũng hợp lệ (`ap3` giữ vị trí 0 và 1: `blade`, `plain` khác ở vị trí 0, `amber` khác ở vị trí 1). Đây đúng là tình huống "nhiều đáp án" mà đề nói; đó là lý do phần test phải kiểm tra **tính hợp lệ + độ dài**, không so chuỗi.

### Trace Cách 3 trên Ví dụ 1

`trace_dfs` in mỗi nút: vị trí `i`, phần viết tắt đã dựng, `length`, và lý do dừng (nếu có). Thụt lề theo độ sâu.

In [17]:
def trace_dfs(target: str, dictionary: List[str]):
    m = len(target)
    diffs = [diff_mask(target, w) for w in dictionary if len(w) == m]
    best = [m + 1, None]
    print("diffs:", [bits(d, m) for d in diffs])

    def dfs(i, mask, length, partial, depth):
        pad = "  " * depth
        if length >= best[0]:
            print(f"{pad}i={i} {partial!r:7} len={length} -> CẮT (len >= best_len={best[0]})")
            return
        if i == m:
            ok = all(mask & d for d in diffs)
            if ok:
                best[0], best[1] = length, partial
            print(f"{pad}i={i} {partial!r:7} len={length} -> lá: {'HỢP LỆ, best=' + repr(partial) if ok else 'khớp từ điển, bỏ'}")
            return
        low = (1 << i) - 1
        if any(d & low == d and mask & d == 0 for d in diffs):
            print(f"{pad}i={i} {partial!r:7} len={length} -> CẮT (nhánh chết)")
            return
        print(f"{pad}i={i} {partial!r:7} len={length}")
        dfs(i + 1, mask | (1 << i), length + 1, partial + target[i], depth + 1)
        for j in range(i + 1, m + 1):
            if j == m:
                dfs(m, mask, length + 1, partial + str(j - i), depth + 1)
            else:
                dfs(j + 1, mask | (1 << j), length + 2, partial + str(j - i) + target[j], depth + 1)

    dfs(0, 0, 0, "", 0)
    print("Kết quả:", best[1])


trace_dfs("apple", ["blade"])

diffs: ['11110']
i=0 ''      len=0
  i=1 'a'     len=1
    i=2 'ap'    len=2
      i=3 'app'   len=3
        i=4 'appl'  len=4
          i=5 'apple' len=5 -> lá: HỢP LỆ, best='apple'
          i=5 'appl1' len=5 -> CẮT (len >= best_len=5)
        i=5 'app1e' len=5 -> CẮT (len >= best_len=5)
        i=5 'app2'  len=4 -> lá: HỢP LỆ, best='app2'
      i=4 'ap1l'  len=4 -> CẮT (len >= best_len=4)
      i=5 'ap2e'  len=4 -> CẮT (len >= best_len=4)
      i=5 'ap3'   len=3 -> lá: HỢP LỆ, best='ap3'
    i=3 'a1p'   len=3 -> CẮT (len >= best_len=3)
    i=4 'a2l'   len=3 -> CẮT (len >= best_len=3)
    i=5 'a3e'   len=3 -> CẮT (len >= best_len=3)
    i=5 'a4'    len=2 -> lá: HỢP LỆ, best='a4'
  i=2 '1p'    len=2 -> CẮT (len >= best_len=2)
  i=3 '2p'    len=2 -> CẮT (len >= best_len=2)
  i=4 '3l'    len=2 -> CẮT (len >= best_len=2)
  i=5 '4e'    len=2 -> CẮT (len >= best_len=2)
  i=5 '5'     len=1 -> lá: khớp từ điển, bỏ
Kết quả: a4


Đọc trace (khớp output ở trên):

| Giai đoạn | Nút | `best_len` sau | Điều xảy ra |
|---|---|---|---|
| Lặn sâu nhánh "giữ chữ" | `a → ap → app → appl → apple` | 5 | Lá đầu tiên `"apple"` hợp lệ (luôn hợp lệ vì `target` không có trong từ điển) |
| Quay lui tầng `appl` | `appl1` (len 5) | 5 | **CẮT**: 5 ≥ 5, không ngắn hơn |
| Quay lui tầng `app` | `app1e` cắt; `app2` (len 4) | 4 | `"app2"` hợp lệ → `best_len = 4` |
| Quay lui tầng `ap` | `ap1l`, `ap2e` cắt; `ap3` (len 3) | 3 | `"ap3"` hợp lệ → `best_len = 3` |
| Quay lui tầng `a` | `a1p`, `a2l`, `a3e` cắt; `a4` (len 2) | 2 | `"a4"` hợp lệ → `best_len = 2` |
| Nhánh "gộp từ đầu" | `1p`, `2p`, `3l`, `4e` | 2 | Đều có len 2 ≥ 2 → **CẮT** ngay, không đi sâu |
| Lá `5` | `5` (len 1) | 2 | Ngắn hơn nhưng `mask = 0` không chạm diff `11110` → khớp `blade`, bỏ |

Toàn bộ chỉ cần **21 nút**. Ở đây chưa thấy Luật 2 hoạt động vì diff duy nhất `11110` có bit ở vị trí 3 — chỉ "chết" khi đã đi qua vị trí 3 mà chưa giữ chữ nào trong 0..3; mọi nhánh như vậy đã bị Luật 1 cắt trước. Ở ca `m = 21` dưới đây, hai luật cùng nhau chỉ cần **253 nút** thay vì `2^21` mask. Đáp án `"a4"` đúng như đề.

### Vì sao cắt tỉa **không bao giờ** làm mất đáp án?

- Luật 1: `length` của một nút là độ dài phần đã dựng; mọi viết tắt hoàn chỉnh trong nhánh đó có độ dài **≥** con số này. Nếu nó đã `>= best_len`, nhánh không chứa viết tắt nào **ngắn hơn** đáp án đang có → bỏ không mất gì (ta chỉ cần **một** đáp án ngắn nhất).
- Luật 2: các vị trí `< i` không bao giờ bị thay đổi nữa trong nhánh. Nếu một từ có mọi vị trí khác `target` nằm ở đó mà chưa được giữ vị trí nào, mọi lá trong nhánh đều khớp từ ấy → không lá nào hợp lệ.

In [18]:
import random
import time

random.seed(411)
m = 21
target = "".join(random.choice("ab") for _ in range(m))
opposite = "".join("b" if c == "a" else "a" for c in target)       # khác target ở MỌI vị trí

for name, fn in [("Cách 2 (duyệt 2^m)", min_abbr_diffmask)]:
    t0 = time.perf_counter()
    ans = fn(target, [opposite])
    print(f"{name}: {ans!r}  ~{time.perf_counter() - t0:.2f}s")
st = {}
t0 = time.perf_counter()
ans = min_abbr_dfs(target, [opposite], st)
print(f"Cách 3 (DFS + cắt tỉa): {ans!r}  ~{time.perf_counter() - t0:.4f}s, số nút = {st['visited']}  (so với 2^21 = {2**21})")
print("(thời gian đo thay đổi theo máy; số nút thì cố định)")

Cách 2 (duyệt 2^m): 'a20'  ~0.32s
Cách 3 (DFS + cắt tỉa): 'a20'  ~0.0003s, số nút = 253  (so với 2^21 = 2097152)
(thời gian đo thay đổi theo máy; số nút thì cố định)


### Độ phức tạp Cách 3

- **Tệ nhất** vẫn là hàm mũ: số nút DFS không vượt quá số viết tắt cộng các nút trung gian, cỡ `O(2^m)`, mỗi nút tốn `O(n')` cho Luật 2 và `O(m)` cho vòng `for j`. Không có cận tốt hơn trong trường hợp tổng quát — bài toán "hitting set" nói chung là NP-khó, nên không ai kỳ vọng thuật toán đa thức.
- **Thực tế** nhanh hơn Cách 2 rất nhiều vì đáp án thường ngắn, Luật 1 cắt rất sớm.
- **Bộ nhớ:** `O(m)` cho ngăn xếp đệ quy (độ sâu ≤ `m + 1`) + `O(n')` cho `diffs`.

## 7. Cách 4 (tùy chọn) — Chỉ duyệt mask con của `U`, bỏ diff-mask thừa

### Quan sát 1: bỏ bớt một chữ đang giữ **không bao giờ** làm viết tắt dài thêm

Giả sử vị trí `i` đang được giữ (bit 1), ta đổi nó thành bị gộp (bit 0). Số chữ giữ giảm 1. Số khối số thay đổi tùy hai hàng xóm:

| Hai bên vị trí `i` | Ví dụ (trước → sau) | Số khối | Số chữ | Độ dài thay đổi |
|---|---|---|---|---|
| cả hai bên là 0 | `0 1 0` → `0 0 0` (`1x1` → `3`) | −1 (hai khối nhập làm một) | −1 | **−2** |
| đúng một bên là 0 | `0 1 1` → `0 0 1` (`1xy` → `2y`) | 0 (nhập vào khối bên cạnh) | −1 | **−1** |
| không bên nào là 0 (hoặc là biên) | `1 1 1` → `1 0 1` (`xyz` → `x1z`) | +1 (khối mới) | −1 | **0** |

→ Độ dài **không tăng**. Hệ quả: gọi `U = OR của mọi diff`. Giữ một chữ ở vị trí **ngoài** `U` không giúp chạm thêm `diff` nào (mọi từ đều giống `target` ở đó). Với mọi mask hợp lệ `K`, mask `K & U` vẫn hợp lệ (vì `(K & U) & d = K & d` khi `d ⊆ U`) và **không dài hơn**. Vậy chỉ cần duyệt các mask là **tập con của `U`**: `2^{|U|}` mask thay vì `2^m`.

### Quan sát 2: diff-mask "thừa"

Nếu `d1 ⊆ d2` (tức `d1 & d2 == d1`), mọi mask chạm `d1` thì cũng chạm `d2` → bỏ `d2` cũng được. Trùng lặp cũng bỏ. Ít `diff` hơn → kiểm tra `all(...)` nhanh hơn.

Cell dưới kiểm chứng Quan sát 1 bằng thực nghiệm trên mọi mask với `m = 8`.

In [19]:
m = 8
for mask in range(1 << m):
    for i in range(m):
        if mask >> i & 1:
            smaller = mask & ~(1 << i)               # bỏ giữ vị trí i
            assert abbr_len_fast(smaller, m) <= abbr_len_fast(mask, m)
print("Bỏ một chữ đang giữ không bao giờ làm tăng độ dài (kiểm trên mọi mask m = 8)")
print("Ví dụ:", mask_to_abbr("abcde", 0b10101), abbr_len_fast(0b10101, 5), "->", mask_to_abbr("abcde", 0b10001), abbr_len_fast(0b10001, 5))

Bỏ một chữ đang giữ không bao giờ làm tăng độ dài (kiểm trên mọi mask m = 8)
Ví dụ: a1c1e 5 -> a3e 3


### Kiến thức nền: duyệt mọi **mask con** của `U` bằng `sub = (sub - 1) & U`

Muốn liệt kê mọi tập con của `U` (không phải của cả `{0..m-1}`), có một mẹo kinh điển:

```python
sub = U
while True:
    ...xử lý sub...
    if sub == 0:
        break
    sub = (sub - 1) & U
```

Vì sao đúng? `sub - 1` lật bit 1 **thấp nhất** của `sub` thành 0 và biến mọi bit thấp hơn thành 1; sau đó `& U` giữ lại đúng những bit được phép. Kết quả là mask con **lớn nhất** của `U` mà nhỏ hơn `sub`. Vòng lặp đi qua mọi mask con theo thứ tự giảm dần, mỗi cái đúng một lần, và **kết thúc ở 0** (nhớ xử lý cả `sub = 0` trước khi `break`!).

Ví dụ `U = 0b1011` (ở đây in kiểu nhị phân thường, bit cao bên trái) có `2^3 = 8` mask con.

In [20]:
U = 0b1011
sub = U
seen = []
while True:
    seen.append(format(sub, "04b"))
    if sub == 0:
        break
    sub = (sub - 1) & U
print(seen)
print("số mask con:", len(seen), "= 2 **", U.bit_count())


def min_abbr_submask(target: str, dictionary: List[str]) -> str:
    m = len(target)
    diffs = sorted({diff_mask(target, w) for w in dictionary if len(w) == m}, key=int.bit_count)
    kept = []
    for d in diffs:                                   # bỏ diff là "tập cha" của một diff đã giữ
        if not any(k & d == k for k in kept):
            kept.append(d)
    U = 0
    for d in kept:
        U |= d
    best_len, best_mask = m + 1, (1 << m) - 1
    sub = U
    while True:
        length = abbr_len_fast(sub, m)
        if length < best_len and all(sub & d for d in kept):
            best_len, best_mask = length, sub
        if sub == 0:
            break
        sub = (sub - 1) & U
    return mask_to_abbr(target, best_mask)


for target, dictionary, expected in examples:
    print(f"{target!r} {dictionary} -> {min_abbr_submask(target, dictionary)!r}")

target = "abcdefghijkl"
dictionary = ["abcdefghizkz", "abcdefghijzz", "abcdefghizkl"]
U = 0
for w in dictionary:
    U |= diff_mask(target, w)
print("U =", bits(U, len(target)), "-> chỉ duyệt", 2 ** U.bit_count(), "mask thay vì", 2 ** len(target))
a4, a2 = min_abbr_submask(target, dictionary), min_abbr_diffmask(target, dictionary)
print(f"Cách 4: {a4!r} (độ dài {abbr_length_of_string(a4)}) | Cách 2: {a2!r} (độ dài {abbr_length_of_string(a2)})  -> khác chuỗi, cùng độ dài")

['1011', '1010', '1001', '1000', '0011', '0010', '0001', '0000']
số mask con: 8 = 2 ** 3
'apple' ['blade'] -> 'a4'
'apple' ['blade', 'plain', 'amber'] -> '3le'
U = 000000000111 -> chỉ duyệt 8 mask thay vì 4096
Cách 4: '9jkl' (độ dài 4) | Cách 2: '9jk1' (độ dài 4)  -> khác chuỗi, cùng độ dài


Lưu ý: Cách 4 có ích khi các từ trong từ điển **giống `target` ở nhiều vị trí** (`U` nhỏ). Nếu có một từ khác `target` ở mọi vị trí thì `U = full` và Cách 4 không bớt được mask nào (còn chậm hơn Cách 2 một chút vì vòng `while` thủ công). Nó là một mẹo đáng biết — đặc biệt kỹ thuật `(sub - 1) & U` dùng lại ở rất nhiều bài bitmask — nhưng không bắt buộc cho bài này.

## 8. Code nộp LeetCode

Bản chính dùng **Cách 2** (diff-mask + duyệt mọi mask + tính độ dài nhanh): ngắn, dễ viết đúng, tệ nhất `2^21` vòng lặp nhẹ. Mọi hàm phụ được viết gọn bên trong `minAbbreviation`.

Cell dưới có **hai** bản: `Solution` (Cách 2) và `SolutionDFS` (Cách 3, nhanh hơn trong thực tế) — nộp bản nào cũng được. LeetCode đã `from typing import List` sẵn; khi chạy ngoài LeetCode (như notebook này) ta tự import. `int.bit_count()` cần Python ≥ 3.10 (LeetCode dùng Python 3 bản mới); nếu môi trường cũ, thay bằng `bin(x).count("1")`.

In [21]:
from typing import List


class Solution:
    def minAbbreviation(self, target: str, dictionary: List[str]) -> str:
        m = len(target)
        # 1) diff-mask của các từ cùng độ dài (từ khác độ dài không bao giờ khớp)
        diffs = []
        for word in dictionary:
            if len(word) != m:
                continue
            d = 0
            for i in range(m):
                if word[i] != target[i]:
                    d |= 1 << i
            diffs.append(d)

        # 2) duyệt mọi mask "giữ chữ", giữ mask hợp lệ có độ dài nhỏ nhất
        full = (1 << m) - 1
        best_len, best_mask = m + 1, full          # full luôn hợp lệ vì target không có trong dictionary
        for mask in range(1 << m):
            zeros = ~mask & full
            length = mask.bit_count() + (zeros & ~(zeros << 1)).bit_count()
            if length < best_len and all(mask & d for d in diffs):
                best_len, best_mask = length, mask

        # 3) dựng chuỗi viết tắt từ best_mask
        out, cnt = [], 0
        for i, ch in enumerate(target):
            if best_mask >> i & 1:
                if cnt:
                    out.append(str(cnt))
                    cnt = 0
                out.append(ch)
            else:
                cnt += 1
        if cnt:
            out.append(str(cnt))
        return "".join(out)


# Bản thứ hai (nhanh hơn trong thực tế) dùng Cách 3 — DFS + cắt tỉa. Nộp bản nào cũng được.
class SolutionDFS:
    def minAbbreviation(self, target: str, dictionary: List[str]) -> str:
        m = len(target)
        diffs = []
        for word in dictionary:
            if len(word) == m:
                d = 0
                for i in range(m):
                    if word[i] != target[i]:
                        d |= 1 << i
                diffs.append(d)
        self.best_len, self.best_mask = m + 1, (1 << m) - 1

        def dfs(i, mask, length):
            if length >= self.best_len:
                return
            if i == m:
                if all(mask & d for d in diffs):
                    self.best_len, self.best_mask = length, mask
                return
            low = (1 << i) - 1
            for d in diffs:
                if d & low == d and mask & d == 0:
                    return
            dfs(i + 1, mask | (1 << i), length + 1)
            for j in range(i + 1, m + 1):
                if j == m:
                    dfs(m, mask, length + 1)
                else:
                    dfs(j + 1, mask | (1 << j), length + 2)

        dfs(0, 0, 0)
        out, cnt = [], 0
        for i, ch in enumerate(target):
            if self.best_mask >> i & 1:
                if cnt:
                    out.append(str(cnt))
                    cnt = 0
                out.append(ch)
            else:
                cnt += 1
        if cnt:
            out.append(str(cnt))
        return "".join(out)

### Test khi có nhiều đáp án đúng

Vì đề cho phép trả **bất kỳ** viết tắt ngắn nhất nào, ta viết hàm kiểm tra `check_answer(target, dictionary, ans)` dựa hoàn toàn vào hàm #408 (độc lập với bitmask, nên không "tự chấm bài mình"):

1. `ans` là viết tắt hợp lệ của `target`: `valid_word_abbreviation(target, ans)`. Hàm #408 đã chặn số có `0` đứng đầu; còn "hai khối số kề nhau" thì không thể xuất hiện trong một chuỗi (các chữ số liên tiếp luôn được đọc thành **một** số — ví dụ `"1111e"` bị đọc là 1111, nên hàm #408 tự trả `False`).
2. `ans` **không** là viết tắt của từ nào trong `dictionary`.
3. Độ dài viết tắt của `ans` bằng độ dài của đáp án brute force (Cách 1).

In [22]:
def check_answer(target: str, dictionary: List[str], ans: str, expected_len: int = None) -> None:
    assert valid_word_abbreviation(target, ans), ("không phải viết tắt của target", ans)
    assert not any(valid_word_abbreviation(w, ans) for w in dictionary), ("khớp một từ trong dictionary", ans)
    if expected_len is None:
        expected_len = abbr_length_of_string(min_abbr_bruteforce(target, dictionary))
    assert abbr_length_of_string(ans) == expected_len, ("chưa ngắn nhất", ans, expected_len)


sol, sol_dfs = Solution(), SolutionDFS()

# Ví dụ 1: chỉ có một đáp án độ dài 2 dùng được -> so chính xác
assert sol.minAbbreviation("apple", ["blade"]) == "a4"
assert sol_dfs.minAbbreviation("apple", ["blade"]) == "a4"

cases = [
    ("apple", ["blade", "plain", "amber"]),          # Ví dụ 2: nhiều đáp án độ dài 3
    ("apple", []),                                    # n = 0 -> "5"
    ("apple", ["applepie", "app", "apples"]),         # chỉ có từ khác độ dài -> "5"
    ("a", ["b"]),                                     # m = 1 -> "a"
    ("a", []),                                        # m = 1, n = 0 -> "1"
    ("ab", ["ac", "cb"]),                             # phải giữ cả 2 chữ -> "ab"
    ("abcdefghijkl", ["abcdefghizkz"]),               # khối số 2 chữ số: "11l"
    ("trace", ["track", "grace", "trick", "bread", "traces"]),
    ("aaaa", ["aaab", "aaba", "abaa", "baaa"]),        # phải giữ mọi chữ
    ("apple", ["blade", "blade", "plain"]),            # từ trùng lặp
]
for target, dictionary in cases:
    for s in (sol, sol_dfs):
        ans = s.minAbbreviation(target, dictionary)
        check_answer(target, dictionary, ans)
    print(f"{target!r:15} {str(dictionary):45} -> {sol.minAbbreviation(target, dictionary)!r:8} DFS: {sol_dfs.minAbbreviation(target, dictionary)!r}")
print("Tất cả test PASS")

'apple'         ['blade', 'plain', 'amber']                   -> '1p3'    DFS: 'ap3'
'apple'         []                                            -> '5'      DFS: '5'
'apple'         ['applepie', 'app', 'apples']                 -> '5'      DFS: '5'
'a'             ['b']                                         -> 'a'      DFS: 'a'
'a'             []                                            -> '1'      DFS: 'a'
'ab'            ['ac', 'cb']                                  -> 'ab'     DFS: 'ab'
'abcdefghijkl'  ['abcdefghizkz']                              -> '11l'    DFS: '11l'
'trace'         ['track', 'grace', 'trick', 'bread', 'traces'] -> 't3e'    DFS: 't3e'
'aaaa'          ['aaab', 'aaba', 'abaa', 'baaa']              -> 'aaaa'   DFS: 'aaaa'
'apple'         ['blade', 'blade', 'plain']                   -> 'a4'     DFS: 'a4'
Tất cả test PASS


## 9. Test đối chiếu chéo trên input ngẫu nhiên

Sinh ngẫu nhiên (cố định `random.seed` để chạy lại ra y hệt) nhiều bộ `target` ngắn (bảng chữ cái nhỏ `"ab"`/`"abc"` để các từ hay trùng chữ → nhiều ràng buộc thú vị) và `dictionary` gồm cả từ cùng độ dài lẫn khác độ dài. Với mỗi bộ, **mọi cách** phải cho đáp án hợp lệ có **cùng độ dài** với brute force.

In [23]:
random.seed(2024)
methods = {
    "diffmask": min_abbr_diffmask,
    "by_length": min_abbr_by_length,
    "dfs": min_abbr_dfs,
    "submask": min_abbr_submask,
    "Solution": Solution().minAbbreviation,
    "SolutionDFS": SolutionDFS().minAbbreviation,
}
n_tests = 0
for _ in range(400):
    alphabet = random.choice(["ab", "abc"])
    m = random.randint(1, 8)
    target = "".join(random.choice(alphabet) for _ in range(m))
    dictionary = []
    for _ in range(random.randint(0, 7)):
        length = m if random.random() < 0.8 else random.randint(1, 9)
        w = "".join(random.choice(alphabet) for _ in range(length))
        if w != target:                          # constraint: dictionary không chứa target
            dictionary.append(w)
    best_len = abbr_length_of_string(min_abbr_bruteforce(target, dictionary))
    for name, fn in methods.items():
        check_answer(target, dictionary, fn(target, dictionary), best_len)
    n_tests += 1
print(f"{n_tests} bộ test ngẫu nhiên: mọi cách đều hợp lệ và ngắn nhất")


# Test ở biên constraints: log2(n) + m <= 21 (không chạy brute force vì quá chậm; so độ dài giữa các cách)
random.seed(7)
big_cases = []
target = "".join(random.choice("ab") for _ in range(21))
big_cases.append((target, ["".join("b" if c == "a" else "a" for c in target)]))       # m = 21, n = 1
target = "".join(random.choice("ab") for _ in range(11))
big_cases.append((target, [w for w in ("".join(random.choice("ab") for _ in range(11)) for _ in range(1000)) if w != target]))  # m = 11, n ≈ 1000
target = "".join(random.choice("abc") for _ in range(16))
big_cases.append((target, ["".join(random.choice("abc") for _ in range(16)) for _ in range(32)]))  # m = 16, n = 32

for target, dictionary in big_cases:
    answers = {name: methods[name](target, dictionary) for name in ["Solution", "SolutionDFS", "submask"]}
    lengths = {abbr_length_of_string(a) for a in answers.values()}
    assert len(lengths) == 1, answers
    best_len = lengths.pop()
    for a in answers.values():
        check_answer(target, dictionary, a, best_len)       # hợp lệ + không khớp từ nào
    print(f"m={len(target):2}, n={len(dictionary):4}: độ dài ngắn nhất = {abbr_length_of_string(answers['Solution'])}, ví dụ đáp án {answers['Solution']!r}")

400 bộ test ngẫu nhiên: mọi cách đều hợp lệ và ngắn nhất


m=21, n=   1: độ dài ngắn nhất = 2, ví dụ đáp án 'b20'
m=11, n=1000: độ dài ngắn nhất = 10, ví dụ đáp án 'baaa1b2aba'
m=16, n=  32: độ dài ngắn nhất = 4, ví dụ đáp án '13caa'


## 10. Vì sao thuật toán đúng? (Cách 2 / `Solution`)

Chuỗi lập luận, mỗi bước đã được kiểm chứng bằng code ở trên:

1. **Mask ↔ viết tắt là 1–1** (mục 2.4): duyệt `range(1 << m)` là duyệt **đủ và không trùng** mọi viết tắt của `target`.
2. **Từ khác độ dài không bao giờ khớp** (mục 4.1): bỏ chúng không làm sai kết luận "hợp lệ / không hợp lệ".
3. **Khớp ⇔ `mask & diff == 0`** (mục 4.2–4.3): nên `all(mask & d for d in diffs)` đúng bằng "không khớp từ nào".
4. **`abbr_len_fast` = độ dài viết tắt** (mục 4.4, kiểm chứng trên mọi mask `m ≤ 10`).
5. **Bất biến (invariant) của vòng lặp:** sau khi xét xong các mask `0..k`, `best_mask` là mask hợp lệ có độ dài nhỏ nhất trong số các mask đã xét **có độ dài < m + 1**, hoặc vẫn là `full` nếu chưa gặp mask hợp lệ nào. Điều kiện `length < best_len` chỉ bỏ qua những mask **không ngắn hơn** đáp án hiện tại → không bỏ sót đáp án ngắn hơn.
6. Mask `full` hợp lệ và có độ dài `m < m + 1`, nên vòng lặp chắc chắn gặp ít nhất một mask hợp lệ → kết thúc với một đáp án ngắn nhất.

Lưu ý: vì dùng `<` (không phải `<=`), khi nhiều mask cùng độ dài ngắn nhất, `Solution` trả về mask **nhỏ nhất về giá trị số** trong số đó — một trong các đáp án hợp lệ, đúng yêu cầu "trả về bất kỳ".

## 11. Độ phức tạp và so sánh các cách

`m = len(target)`, `n = len(dictionary)`, `n'` = số từ cùng độ dài, `|U|` = số bit của hợp các diff.

| Cách | Ý tưởng | Thời gian (tệ nhất) | Bộ nhớ | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | Sinh `2^m` chuỗi, sắp, kiểm tra bằng #408 | `O(2^m · (m log 2^m + n · m))` | `O(2^m · m)` | Chắc chắn đúng; làm trọng tài |
| 2. Diff-mask (**chính**) | `all(mask & d)` + độ dài bằng phép bit | `O(n' · m + 2^m · n')` | `O(n')` | Đơn giản, đủ nhanh với `2^m · n ≤ 2^21` |
| 2b. Theo độ dài tăng dần | Sắp mask theo độ dài, dừng sớm | `O(2^m · m + 2^m · n')` | `O(2^m)` | Dừng sớm nhưng tốn bộ nhớ |
| 3. DFS + cắt tỉa | Branch and bound trên viết tắt | Tệ nhất `O(2^m · (n' + m))`, thực tế rất nhanh | `O(m + n')` | Nhanh nhất trong thực tế |
| 4. Mask con của `U` | Chỉ giữ chữ ở vị trí "phân biệt" | `O(2^{|U|} · n')` | `O(n')` | Có lợi khi các từ giống `target` nhiều |

(Ở Cách 2 coi `bit_count` trên số ≤ 21 bit là `O(1)`.)

## 12. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `bin(mask)` để đọc "vị trí giữ" — ngược thứ tự và mất số 0 đầu

`bin(mask)` in **bit cao trước** và **bỏ số 0 ở đầu**. Nếu ghép trực tiếp với `target` sẽ lệch vị trí.

### Lỗi 2: Quên `& full` khi lấy phần bù `~mask`

Trong Python, số nguyên không giới hạn số bit, nên `~mask` = `-mask - 1` là số **âm** — hình dung là có "vô hạn bit 1" phía trên bit `m - 1`. Khi chữ **cuối** được giữ (bit `m - 1` = 1), dãy bit 1 vô hạn đó bắt đầu ngay tại vị trí `m` và bị đếm nhầm là **thêm một khối số**. Ví dụ `"4e"` (độ dài đúng 2) bị tính thành 3. Cell dưới minh họa cả Lỗi 1 và Lỗi 2.

In [24]:
target = "apple"
mask = 0b00001                      # theo quy ước: giữ vị trí 0 -> "a4"

wrong = bin(mask)[2:]               # '1' -> chỉ 1 ký tự, và nếu có nhiều bit thì ngược thứ tự
print("bin:", wrong, "| zfill:", wrong.zfill(5), "-> nếu đọc trái->phải sẽ tưởng giữ vị trí 4 ('4e')")
print("đúng (bits theo vị trí):", bits(mask, 5), "->", mask_to_abbr(target, mask))
# Sửa: luôn lấy bit i bằng (mask >> i) & 1, hoặc đảo chuỗi: bin(mask)[2:].zfill(m)[::-1]
assert bin(mask)[2:].zfill(5)[::-1] == bits(mask, 5)
print()


# ---------- Lỗi 2 ----------
def abbr_len_wrong(mask: int, m: int) -> int:
    zeros = ~mask                                  # SAI: thiếu & full
    starts = zeros & ~(zeros << 1)
    return mask.bit_count() + starts.bit_count()


mask, m = 0b10000, 5                               # "4e" (giữ chữ cuối), độ dài đúng là 2
print("Lỗi 2 -> sai :", abbr_len_wrong(mask, m), "| ~mask =", ~mask)
print("Lỗi 2 -> đúng:", abbr_len_fast(mask, m))
print("mask không giữ chữ cuối thì bản sai tình cờ vẫn ra đúng:", abbr_len_wrong(0b00010, 5), "=", abbr_len_fast(0b00010, 5))

bin: 1 | zfill: 00001 -> nếu đọc trái->phải sẽ tưởng giữ vị trí 4 ('4e')
đúng (bits theo vị trí): 10000 -> a4

Lỗi 2 -> sai : 3 | ~mask = -17
Lỗi 2 -> đúng: 2
mask không giữ chữ cuối thì bản sai tình cờ vẫn ra đúng: 3 = 3


### Lỗi 3: Tính diff bằng `zip` mà không lọc độ dài

`zip` **âm thầm cắt** theo chuỗi ngắn hơn. Với `target = "apple"` và từ `"applesauce"`, `zip` chỉ so 5 chữ đầu → diff = 0 → không mask nào "chạm" được → thuật toán tưởng không có đáp án (hoặc trả bừa `"apple"`). Thật ra `"applesauce"` khác độ dài nên **không bao giờ** khớp; đáp án đúng là `"5"`.

In [25]:
def min_abbr_wrong_zip(target, dictionary):
    m = len(target)
    diffs = []
    for w in dictionary:                                   # SAI: không lọc len(w) == m
        d = 0
        for i, (a, b) in enumerate(zip(target, w)):
            if a != b:
                d |= 1 << i
        diffs.append(d)
    best_len, best_mask = m + 1, (1 << m) - 1
    for mask in range(1 << m):
        length = abbr_len_fast(mask, m)
        if length < best_len and all(mask & d for d in diffs):
            best_len, best_mask = length, mask
    return mask_to_abbr(target, best_mask)


print("sai :", min_abbr_wrong_zip("apple", ["applesauce"]))
print("đúng:", min_abbr_diffmask("apple", ["applesauce"]))

sai : apple
đúng: 5


### Lỗi 4: Đo độ dài bằng `len(chuỗi)` thay vì "chữ + khối số"

`len("11l")` là 3 nhưng độ dài viết tắt chỉ là 2. Chọn "ngắn nhất" theo `len()` sẽ sai khi xuất hiện số có từ 2 chữ số trở lên (target dài ≥ 10). Ví dụ dưới: target 12 chữ, từ điển có `"abcdefghizkz"` (khác ở vị trí 9 và 11).

In [26]:
def min_abbr_wrong_len(target, dictionary):
    m = len(target)
    diffs = [diff_mask(target, w) for w in dictionary if len(w) == m]
    best = None
    for mask in range(1 << m):
        if all(mask & d for d in diffs):
            s = mask_to_abbr(target, mask)
            if best is None or len(s) < len(best):        # SAI: so len() của chuỗi
                best = s
    return best


target, dictionary = "abcdefghijkl", ["abcdefghizkz"]
w = min_abbr_wrong_len(target, dictionary)
r = min_abbr_diffmask(target, dictionary)
print(f"sai : {w!r} -> độ dài viết tắt {abbr_length_of_string(w)}")
print(f"đúng: {r!r} -> độ dài viết tắt {abbr_length_of_string(r)}")

sai : '9j2' -> độ dài viết tắt 3
đúng: '11l' -> độ dài viết tắt 2


### Lỗi 5: Hiểu ngược điều kiện hợp lệ — dùng `any` thay vì `all`

Viết tắt phải phân biệt `target` với **mọi** từ, nên phải chạm **mọi** diff: `all(mask & d ...)`. Dùng `any` nghĩa là "chỉ cần khác **một** từ" → nhận nhầm `"a4"` ở Ví dụ 2, dù `"a4"` khớp `"amber"`.

In [27]:
target, dictionary = "apple", ["blade", "plain", "amber"]
diffs = [diff_mask(target, w) for w in dictionary]
m = len(target)
best_len, best_mask = m + 1, (1 << m) - 1
for mask in range(1 << m):
    length = abbr_len_fast(mask, m)
    if length < best_len and any(mask & d for d in diffs):     # SAI: any
        best_len, best_mask = length, mask
wrong = mask_to_abbr(target, best_mask)
print("sai :", wrong, "| 'amber' cũng có viết tắt này?", valid_word_abbreviation("amber", wrong))
print("đúng:", min_abbr_diffmask(target, dictionary))

sai : a4 | 'amber' cũng có viết tắt này? True
đúng: 1p3


### Lỗi 6: Sinh viết tắt mà mỗi chữ bị bỏ thành một số `"1"` riêng

Nếu thay **từng** chữ bị bỏ bằng `"1"` thì sẽ tạo ra các khối số **kề nhau** như `"11p3"`... thực chất là chuỗi `"1111e"` — sai định dạng (đề cấm khối kề nhau), và khi đọc lại thì `"1111"` thành số **một nghìn một trăm mười một**! Phải **gộp** các chữ bị bỏ liên tiếp vào một biến đếm (như `cnt` trong `mask_to_abbr`).

In [28]:
def mask_to_abbr_wrong(target, mask):
    return "".join(ch if mask >> i & 1 else "1" for i, ch in enumerate(target))   # SAI: không gộp


mask = 0b10000                                   # giữ vị trí 4 -> đúng phải là "4e"
w = mask_to_abbr_wrong("apple", mask)
print("sai :", w, "| #408 có chấp nhận cho 'apple'?", valid_word_abbreviation("apple", w))
print("đúng:", mask_to_abbr("apple", mask), "|", valid_word_abbreviation("apple", mask_to_abbr("apple", mask)))

sai : 1111e | #408 có chấp nhận cho 'apple'? False
đúng: 4e | True


### Lỗi 7: Duyệt mask con bằng `while sub:` — quên mất `sub = 0`

Ở Cách 4, nếu viết `while sub: ...; sub = (sub - 1) & U` thì mask `0` (viết tắt `str(m)`, ví dụ `"5"`) **không bao giờ** được xét. Khi từ điển không có từ cùng độ dài, `U = 0`, vòng lặp không chạy lần nào → trả về `target` nguyên vẹn thay vì `"5"`. Sửa: xử lý `sub` trước, rồi mới `if sub == 0: break`.

In [29]:
def submask_wrong(target, dictionary):
    m = len(target)
    diffs = [diff_mask(target, w) for w in dictionary if len(w) == m]
    U = 0
    for d in diffs:
        U |= d
    best_len, best_mask = m + 1, (1 << m) - 1
    sub = U
    while sub:                                          # SAI: bỏ sót sub = 0
        length = abbr_len_fast(sub, m)
        if length < best_len and all(sub & d for d in diffs):
            best_len, best_mask = length, sub
        sub = (sub - 1) & U
    return mask_to_abbr(target, best_mask)


print("sai :", submask_wrong("apple", ["app"]))
print("đúng:", min_abbr_submask("apple", ["app"]))

sai : apple
đúng: 5


## 13. Mẹo nhận dạng pattern

- **Độ dài chuỗi rất nhỏ (≤ 20–21)** + "chọn một tập con các vị trí" → nghĩ ngay tới **bitmask** và `range(1 << m)` (khoảng 1–2 triệu trạng thái).
- Ràng buộc kiểu `log2(n) + m <= 21` ⇔ `n · 2^m <= 2^21`: đề **gợi ý thẳng** độ phức tạp `O(2^m · n)`.
- "Đối tượng A **khớp** đối tượng B ⇔ A và B đồng ý tại mọi vị trí được chọn" → tính trước **diff-mask** (vị trí khác nhau), khớp ⇔ `chọn & diff == 0`. Mẫu này xuất hiện ở nhiều bài so khớp có ký tự đại diện.
- "Phải phân biệt với **mọi** phần tử" → điều kiện `all(...)`; "chọn ít nhất để chạm mọi tập" → bài **hitting set** → chấp nhận hàm mũ + cắt tỉa.
- "Trả về bất kỳ đáp án ngắn nhất nào" → khi test, viết **hàm kiểm tra** + so **giá trị tối ưu** với brute force, đừng so chuỗi.
- Hàm mục tiêu (độ dài) **chỉ tăng** khi dựng dần → **branch and bound** (cắt khi `cost >= best`).

## 14. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| #408 Valid Word Abbreviation | Hàm kiểm tra dùng xuyên notebook (hai con trỏ + đọc số) |
| #320 Generalized Abbreviation | Sinh mọi viết tắt — bitmask hoặc DFS như mục 2.4 / 6 |
| #288 Unique Word Abbreviation | Viết tắt cố định kiểu "chữ đầu + số + chữ cuối", kiểm tra tính duy nhất bằng `dict` |
| #527 Word Abbreviation | Cho danh sách từ, tìm viết tắt ngắn nhất dạng "tiền tố + số + chữ cuối" cho **mỗi** từ sao cho không trùng nhau |
| #78 Subsets | Duyệt mọi tập con bằng `range(1 << n)` |
| #1178 Number of Valid Words for Each Puzzle | Bitmask + duyệt mask con `(sub - 1) & U` |

Biến thể tự nghĩ (một số có trong phần bài tập):
- Liệt kê **tất cả** viết tắt ngắn nhất hợp lệ.
- Áp #411 cho **từng** từ trong một danh sách (phân biệt mỗi từ với các từ còn lại).
- Đổi định nghĩa độ dài (ví dụ đếm `len()` thật của chuỗi): chỉ cần đổi hàm độ dài, phần diff-mask giữ nguyên.

## 15. Bài tập vận dụng

### Bài 1 — Đoán output (không chạy code)

Với các hàm đã học (`bits`, `mask_to_abbr`, `abbr_len_fast`), đoán kết quả:

```python
a) mask_to_abbr("banana", 0b100001)
b) mask_to_abbr("banana", 0b010010)
c) abbr_len_fast(0b010010, 6)
d) bits(6, 4)
e) bin(6)
f) abbr_len_fast(0, 7)
```

### Bài 2 — Tính tay bằng diff-mask

`target = "trace"`, `dictionary = ["track", "grace", "trick", "bread", "traces"]`.

a) Từ nào bị lọc bỏ ngay? Vì sao?
b) Viết diff-mask (in theo vị trí, trái = vị trí 0) của các từ còn lại.
c) Mask "giữ chữ" phải chứa những vị trí nào? Suy ra đáp án ngắn nhất và độ dài của nó.

### Bài 3 — Điền TODO: tính độ dài viết tắt từ mask

Hoàn thiện `exercise_abbr_len(mask, m)` bằng phép bit (không dựng chuỗi).

In [30]:
def exercise_abbr_len(mask: int, m: int) -> int:
    # TODO 1: full = mặt nạ m bit toàn 1
    # TODO 2: zeros = các vị trí bị gộp (nhớ & full)
    # TODO 3: starts = các vị trí bắt đầu một đoạn 0 liên tiếp
    # TODO 4: trả về số chữ giữ + số đoạn 0
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_abbr_len(0b00010, 5) == 3      # "1p3"
# assert exercise_abbr_len(0b00000, 5) == 1      # "5"
# assert exercise_abbr_len(0b11111, 5) == 5      # "apple"
# assert exercise_abbr_len(0b10101, 5) == 5      # "a1p1e"
# assert all(exercise_abbr_len(k, 8) == abbr_len_fast(k, 8) for k in range(1 << 8))

### Bài 4 — Tự viết lại #408 Valid Word Abbreviation

Không nhìn lên trên, viết `exercise_valid_abbr(word, abbr)`. Nhớ: chữ cái phải khớp; khối số đọc trọn (`num = num * 10 + d`); số bắt đầu bằng `'0'` là sai; cuối cùng **cả hai** con trỏ phải tới cuối.

In [31]:
def exercise_valid_abbr(word: str, abbr: str) -> bool:
    # TODO: hai con trỏ i (word), j (abbr)
    raise NotImplementedError

# Gợi ý test:
# assert exercise_valid_abbr("internationalization", "i12iz4n") == True
# assert exercise_valid_abbr("apple", "a2e") == False
# assert exercise_valid_abbr("substitution", "s010n") == False
# assert exercise_valid_abbr("apple", "6") == False
# assert exercise_valid_abbr("a", "1") == True

### Bài 5 — Biến thể: liệt kê **tất cả** đáp án ngắn nhất

Viết `exercise_all_shortest(target, dictionary)` trả về **danh sách đã sắp xếp** mọi viết tắt hợp lệ có độ dài nhỏ nhất. Ví dụ Ví dụ 1 → `["a4"]`; với Ví dụ 2, danh sách phải chứa cả `"1p3"`, `"2p2"`, `"3l1"` mà đề nhắc tới (và vài cái khác nữa — có bao nhiêu?).

In [32]:
def exercise_all_shortest(target: str, dictionary: List[str]) -> List[str]:
    # TODO: tính diffs; lượt 1 tìm best_len; lượt 2 gom mọi mask hợp lệ có độ dài best_len
    pass

# Gợi ý test:
# assert exercise_all_shortest("apple", ["blade"]) == ["a4"]
# res = exercise_all_shortest("apple", ["blade", "plain", "amber"])
# assert {"1p3", "2p2", "3l1"} <= set(res)
# assert exercise_all_shortest("apple", []) == ["5"]

### Bài 6 — Mini challenge: viết tắt độc nhất cho **từng** từ trong danh sách

Cho danh sách `words` gồm các từ **khác nhau**. Với mỗi từ `words[i]`, tìm viết tắt ngắn nhất của nó **không** là viết tắt của bất kỳ từ nào khác trong danh sách. Viết `exercise_unique_for_each(words)` trả về danh sách kết quả (mỗi vị trí trả bất kỳ đáp án ngắn nhất nào).

Ví dụ `["apple", "blade", "plain", "amber"]`: đáp án cho `"apple"` có độ dài 3 (như Ví dụ 2). Tự kiểm tra độ dài các đáp án còn lại bằng `check_answer`.

Câu hỏi thêm: nếu `words` có `k` từ cùng độ dài `m`, tổng thời gian của cách bạn viết là bao nhiêu? Có tính lại diff giữa hai từ hai lần không? (Gợi ý: `diff(a, b) == diff(b, a)`.)

In [33]:
def exercise_unique_for_each(words: List[str]) -> List[str]:
    # TODO: với mỗi i, gọi lời giải #411 với target = words[i], dictionary = các từ còn lại
    pass

# Gợi ý test:
# words = ["apple", "blade", "plain", "amber"]
# res = exercise_unique_for_each(words)
# for i, ans in enumerate(res):
#     check_answer(words[i], words[:i] + words[i + 1:], ans)

## 16. Gợi ý theo mức

**Bài 1**
- Mức 1: bit `i` = vị trí `i` của chuỗi; số nhị phân viết bit cao (vị trí lớn) **bên trái**.
- Mức 2: `0b100001` có bit 0 và bit 5; `0b010010` có bit 1 và bit 4.
- Mức 3: `bits` in vị trí 0 trước; `bin` in bit cao trước. Mask 0 → một khối số duy nhất.

**Bài 2**
- Mức 1: từ khác độ dài không bao giờ khớp.
- Mức 2: so từng chữ `t r a c e` với từng từ; ghi 1 ở chỗ khác.
- Mức 3: một diff chỉ có **một** bit 1 → vị trí đó **bắt buộc** được giữ.

**Bài 3**
- Mức 1: độ dài = chữ giữ + số đoạn 0.
- Mức 2: `full = (1 << m) - 1`, `zeros = ~mask & full`.
- Mức 3: `starts = zeros & ~(zeros << 1)`; dùng `.bit_count()`.

**Bài 4**
- Mức 1: hai con trỏ chạy song song, chữ số thì "nhảy", chữ cái thì "so".
- Mức 2: kiểm `'0'` ngay khi bắt đầu một khối số.
- Mức 3: điều kiện trả về là `i == len(word) and j == len(abbr)` (không phải chỉ một trong hai).

**Bài 5**
- Mức 1: làm hai lượt, hoặc một lượt vừa giữ `best_len` vừa gom danh sách (reset khi thấy độ dài nhỏ hơn).
- Mức 2: dùng lại `diff_mask`, `abbr_len_fast`, `mask_to_abbr`.
- Mức 3: nhớ `sorted(...)` ở cuối.

**Bài 6**
- Mức 1: đây chỉ là gọi #411 `k` lần.
- Mức 2: `dictionary = words[:i] + words[i + 1:]`.
- Mức 3: tối ưu: tính diff cho mọi cặp cùng độ dài một lần (`k^2/2` cặp), rồi mỗi từ chạy vòng `2^m`; tổng `O(k^2 · m + k · 2^m · k)`.

## 17. Đáp án tham khảo

### Đáp án Bài 1

- a) `0b100001`: giữ vị trí 0 (`b`) và 5 (`a`) → `"b4a"`.
- b) `0b010010`: giữ vị trí 1 (`a`) và 4 (`n`) → `"1a2n1"`.
- c) `"1a2n1"`: 2 chữ + 3 khối số = `5`.
- d) `6 = 0b0110` → bit 1 và bit 2 → theo vị trí: `"0110"` (trùng hợp đối xứng!).
- e) `'0b110'` (bit cao trước, không có số 0 đầu).
- f) Mask 0 → `"7"` → độ dài `1`.

### Đáp án Bài 2

- a) `"traces"` (6 chữ) bị lọc vì khác độ dài.
- b) `track` → `00001` (chỉ khác ở vị trí 4: `k≠e`); `grace` → `10000` (vị trí 0: `g≠t`); `trick` → `00101` (vị trí 2 và 4); `bread` → `10111` (vị trí 0, 2, 3, 4).
- c) `track` buộc giữ vị trí 4, `grace` buộc giữ vị trí 0; khi đó `trick` (chứa vị trí 4) và `bread` (chứa vị trí 0) cũng được chạm. Mask `10001` → `"t3e"`, độ dài 3. Không thể ngắn hơn vì đã buộc giữ 2 chữ ở hai đầu, ở giữa còn 3 chữ tạo 1 khối.

In [34]:
# Kiểm tra đáp án Bài 1 và Bài 2
assert mask_to_abbr("banana", 0b100001) == "b4a"
assert mask_to_abbr("banana", 0b010010) == "1a2n1"
assert abbr_len_fast(0b010010, 6) == 5
assert bits(6, 4) == "0110"
assert bin(6) == "0b110"
assert abbr_len_fast(0, 7) == 1

target, dictionary = "trace", ["track", "grace", "trick", "bread", "traces"]
print({w: bits(diff_mask(target, w), 5) for w in dictionary if len(w) == 5})
assert min_abbr_diffmask(target, dictionary) == "t3e"
assert min_abbr_bruteforce(target, dictionary) == "t3e"
print("Bài 1, Bài 2: PASS")

{'track': '00001', 'grace': '10000', 'trick': '00101', 'bread': '10111'}
Bài 1, Bài 2: PASS


### Đáp án Bài 3 và Bài 4

Bài 3 chính là `abbr_len_fast` ở mục 4.4. Bài 4 là `valid_word_abbreviation` ở mục 2.1, viết gọn nhánh chữ cái bằng `elif`.

In [35]:
def exercise_abbr_len(mask: int, m: int) -> int:
    full = (1 << m) - 1
    zeros = ~mask & full
    starts = zeros & ~(zeros << 1)
    return mask.bit_count() + starts.bit_count()


assert exercise_abbr_len(0b00010, 5) == 3
assert exercise_abbr_len(0b00000, 5) == 1
assert exercise_abbr_len(0b11111, 5) == 5
assert exercise_abbr_len(0b10101, 5) == 5
assert all(exercise_abbr_len(k, 8) == abbr_len_fast(k, 8) for k in range(1 << 8))
print("Bài 3: PASS")


# ---------- Đáp án Bài 4 ----------
def exercise_valid_abbr(word: str, abbr: str) -> bool:
    i = j = 0
    while i < len(word) and j < len(abbr):
        if abbr[j].isdigit():
            if abbr[j] == "0":
                return False
            num = 0
            while j < len(abbr) and abbr[j].isdigit():
                num = num * 10 + int(abbr[j])
                j += 1
            i += num
        elif word[i] == abbr[j]:
            i += 1
            j += 1
        else:
            return False
    return i == len(word) and j == len(abbr)


assert exercise_valid_abbr("internationalization", "i12iz4n") == True
assert exercise_valid_abbr("apple", "a2e") == False
assert exercise_valid_abbr("substitution", "s010n") == False
assert exercise_valid_abbr("apple", "6") == False
assert exercise_valid_abbr("a", "1") == True
print("Bài 4: PASS")

Bài 3: PASS
Bài 4: PASS


### Đáp án Bài 5

Một lượt duy nhất: khi gặp mask hợp lệ có độ dài **nhỏ hơn** `best_len` thì reset danh sách; **bằng** thì thêm vào.

In [36]:
def exercise_all_shortest(target: str, dictionary: List[str]) -> List[str]:
    m = len(target)
    diffs = [diff_mask(target, w) for w in dictionary if len(w) == m]
    best_len, found = m + 1, []
    for mask in range(1 << m):
        length = abbr_len_fast(mask, m)
        if length > best_len or not all(mask & d for d in diffs):
            continue
        if length < best_len:
            best_len, found = length, []
        found.append(mask_to_abbr(target, mask))
    return sorted(found)


assert exercise_all_shortest("apple", ["blade"]) == ["a4"]
res = exercise_all_shortest("apple", ["blade", "plain", "amber"])
assert {"1p3", "2p2", "3l1"} <= set(res)
assert exercise_all_shortest("apple", []) == ["5"]
print("Ví dụ 2 có", len(res), "đáp án ngắn nhất:", res)
print("Bài 5: PASS")

Ví dụ 2 có 6 đáp án ngắn nhất: ['1p3', '2p2', '3l1', '3le', 'a3e', 'ap3']
Bài 5: PASS


### Đáp án Bài 6

Cách thẳng: gọi `Solution().minAbbreviation` cho từng từ. (Bản tối ưu hơn sẽ tính bảng diff cho mọi cặp một lần, như gợi ý mức 3.)

In [37]:
def exercise_unique_for_each(words: List[str]) -> List[str]:
    sol = Solution()
    return [sol.minAbbreviation(w, words[:i] + words[i + 1:]) for i, w in enumerate(words)]


for words in (["apple", "blade", "plain", "amber"], ["like", "god", "internal", "me", "internet", "interval", "intension", "face", "intrusion"]):
    res = exercise_unique_for_each(words)
    for i, ans in enumerate(res):
        check_answer(words[i], words[:i] + words[i + 1:], ans)
    print(list(zip(words, res)))
print("Bài 6: PASS")

[('apple', '1p3'), ('blade', 'b4'), ('plain', 'p4'), ('amber', '4r')]
[('like', 'l3'), ('god', '3'), ('internal', '5na1'), ('me', '2'), ('internet', '7t'), ('interval', '5v2'), ('intension', '3e5'), ('face', 'f3'), ('intrusion', '3r5')]
Bài 6: PASS


## 18. Template cuối cùng nên nhớ

```python
class Solution:
    def minAbbreviation(self, target: str, dictionary: List[str]) -> str:
        m = len(target)
        diffs = []                                   # chỉ từ CÙNG độ dài
        for w in dictionary:
            if len(w) == m:
                d = 0
                for i in range(m):
                    if w[i] != target[i]:
                        d |= 1 << i                  # bit i = 1: w khác target ở i
                diffs.append(d)
        full = (1 << m) - 1
        best_len, best_mask = m + 1, full            # full luôn hợp lệ
        for mask in range(1 << m):                   # bit 1 = GIỮ chữ
            zeros = ~mask & full                     # nhớ & full
            length = mask.bit_count() + (zeros & ~(zeros << 1)).bit_count()
            if length < best_len and all(mask & d for d in diffs):   # chạm MỌI diff
                best_len, best_mask = length, mask
        out, cnt = [], 0                             # mask -> chuỗi (gộp số 0 liên tiếp)
        for i, ch in enumerate(target):
            if best_mask >> i & 1:
                if cnt: out.append(str(cnt)); cnt = 0
                out.append(ch)
            else:
                cnt += 1
        if cnt: out.append(str(cnt))
        return "".join(out)
```

Câu thần chú: **"Cùng độ dài mới so; khác nhau thành diff; giữ chữ phải chạm mọi diff; độ dài = chữ giữ + số đoạn 0."**

Muốn nhanh hơn: DFS dựng viết tắt trái → phải, cắt khi `length >= best_len` hoặc khi một diff đã nằm hết bên trái mà chưa được chạm (bản `SolutionDFS`).

## 19. Checklist tự kiểm tra trước khi submit

- Đã **lọc** chỉ giữ từ `len(w) == m` chưa? (Không dùng `zip` với từ khác độ dài.)
- Quy ước bit có thống nhất không: bit `i` ↔ vị trí `i`, bit 1 = **giữ** chữ? (Đừng đọc `bin(mask)` trái → phải.)
- Điều kiện hợp lệ là `all(mask & d for d in diffs)` (chạm **mọi** diff), không phải `any`, không phải `== 0`.
- Phần bù có `& full` chưa? (`~mask` trong Python là số âm.)
- Độ dài = `bit_count(mask)` + **số đoạn 0** — không phải `len()` của chuỗi (`"11l"` dài 2).
- Khởi tạo `best_len = m + 1` (để mask `full` chắc chắn được xét) và trường hợp không có từ cùng độ dài trả `str(m)`?
- Khi dựng chuỗi, các chữ bị bỏ liên tiếp có được **gộp** thành một số chưa?
- Nếu dùng mask con `(sub - 1) & U`: có xử lý `sub = 0` chưa?
- Đã test: 2 ví dụ đề (Ví dụ 2 kiểm **hợp lệ + độ dài**, không so chuỗi), `dictionary = []`, chỉ có từ khác độ dài, `m = 1`, target ≥ 10 chữ (số 2 chữ số), từ trùng lặp, và đối chiếu với brute force trên input ngẫu nhiên?